
# Week 3 — Gemma 3 + Shadow-FT on Colab（免費 T4）

**這份 notebook 對應 `week3_執行手冊.md`。手冊講「為什麼」，這裡是「怎麼跑」。**

---

## 開跑前必讀

| 項目 | 值 | 為什麼 |
|---|---|---|
| Runtime | **T4 GPU**（選單：執行階段 → 變更執行階段類型 → T4 GPU） | 免費版唯一選項 |
| 單次上限 | 12 小時 | 中途斷線會全部消失 → **所有產出都寫進 Drive** |
| 每週配額 | 約 15–30 GPU 小時（Google 不公布確切數字，會浮動） | 本 notebook 全跑約 8–10 小時，留有餘裕 |
| bf16 | **沒有**。T4 是 Turing，只有 fp16 | Gemma 3 在 fp16 下 layernorm 後 activation 可達 ~800,000 > fp16 上限 65,504 → `inf` → `NaN`。這就是必須用 Unsloth 的唯一理由 |

**斷點續跑**：每個實驗跑完會在 Drive 寫一個 `done.json`。重跑整份 notebook 時已完成的會自動跳過。所以斷線之後：重連 → 從頭 Run All → 它會接著上次的進度。

---

## 這份 notebook 更新了怎麼辦（它不會自動同步）

Colab 跑的是**它自己的那份**，本機改了不會傳過去。§1.1 會印 `notebook 版本`，
先對一下是不是最新的。

**建議做法：把 notebook 放進同一個 Drive 資料夾，直接從 Drive 開。**

1. 本機把 `notebooks/week3_colab.ipynb` 複製到 Google 雲端硬碟的
   `ultrascale-lab-week3/` 底下（就是這份 notebook 寫結果的同一個地方）
2. 之後都從 雲端硬碟 → 對檔案按右鍵 → 開啟工具 → Google Colaboratory 開
3. 有更新時：覆蓋 Drive 上那個檔 → 回 Colab 分頁按重新整理

這樣「程式碼」和「結果」就在同一個地方，不會出現版本對不上的狀況。

> 如果是用「上傳 notebook」開的，更新就得重新上傳一次。**重新上傳不會弄丟任何進度** ——
> 進度是存在 Drive 的 `results/*.json`，跟 notebook 檔案本身無關。

---

## 執行順序

```
§1 環境      →  §2 資料      →  §3 記憶體預測（先算再量，沿用 Week 1/2 的做法）
     ↓
§4 框架對照（HF vs Unsloth，各 30 步）        ← 答主管 Q1
     ↓
§5 Stage A/B/C：LoRA 參數掃描                 ← 答主管 Q2
     ↓
§6 Shadow-FT（在 -pt 上訓練，把 adapter 搬到 -it）  ← 答主管 Q4
     ↓
§7 TMMLU+ 評測（種子固定、macro + micro 並列）
     ↓
§8 彙整成表與圖
```


---
# §1 環境

## 1.1 確認拿到什麼卡

In [ ]:
#@title 1.1 GPU / 環境檢查
import subprocess, sys, os, platform, json

# 這份 notebook 不會自動同步 —— 本機更新後要重新上傳／覆蓋 Drive 上的檔案。
# 開頭印版本，才不會改了半天發現 Colab 上跑的還是舊版。
NOTEBOOK_VERSION = "2026-08-18g"   # g: σ 與探路也自動跳過，全程不用手動跳格子
print(f"notebook 版本：{NOTEBOOK_VERSION}")

# 【必須在 import torch 之前設】T4 只有 14.56 GiB，而 262K vocab 的 logits 是
# 一整塊大配置。預設的 allocator 會產生大量「保留但未使用」的碎片
# （OOM 訊息裡的 "1.19 GiB is reserved but unallocated" 就是它）。
# expandable_segments 讓 allocator 可以擴張既有 segment，而不是每次都要一塊
# 連續的新記憶體 —— 對「少數幾個超大張量」這種形狀特別有效。
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

try:
    import torch
    p = torch.cuda.get_device_properties(0)
    cc = f"{p.major}.{p.minor}"
    print(f"GPU            : {p.name}")
    print(f"VRAM           : {p.total_memory/2**30:.2f} GiB")
    print(f"Compute cap.   : {cc}")
    print(f"支援 bf16      : {torch.cuda.is_bf16_supported()}")
    print(f"torch          : {torch.__version__}")
    if not torch.cuda.is_bf16_supported():
        print()
        print("→ 沒有 bf16（T4 / Turing）。Gemma 3 在純 fp16 下會溢位成 NaN，")
        print("  必須用 Unsloth 的混合精度修補。§4 會實測給你看。")
except Exception as e:
    print("!! 沒有 GPU：執行階段 → 變更執行階段類型 → T4 GPU")
    raise

In [ ]:
#@title 1.1b RAM 監看（免費版只有約 12.7 GB 系統 RAM，比 GPU 還容易先爆）
import os, gc, subprocess

def ram():
    #「系統 RAM」不是 GPU VRAM。Colab 免費版約 12.7 GB，
    # 而 gemma-3-4b 的 fp16 權重單一份就 8.6 GB —— 一次讀兩個必爆。
    try:
        import psutil
        v = psutil.virtual_memory()
        return v.used / 2**30, v.total / 2**30
    except ImportError:
        with open('/proc/meminfo') as f:
            m = {l.split(':')[0]: int(l.split()[1]) for l in f}
        tot = m['MemTotal'] / 2**20
        avail = m['MemAvailable'] / 2**20
        return tot - avail, tot

def print_ram(tag=""):
    u, t = ram()
    bar = "█" * int(20 * u / t) + "·" * (20 - int(20 * u / t))
    print(f"  RAM {bar} {u:.1f}/{t:.1f} GiB{('  ' + tag) if tag else ''}")
    if u / t > 0.85:
        print("  ⚠️ 超過 85%，下一個大張量可能就會讓 kernel 重啟。先 gc + del 不用的東西。")

def free_ram(*objs):
    for o in objs:
        try: del o
        except Exception: pass
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception: pass

print_ram("目前")


## 1.2 安裝

Unsloth 的相依樹在 Colab 上很敏感，照官方建議的兩段式裝法（先裝完整版拉相依，再用 `--no-deps` 覆蓋成最新）。

In [ ]:
#@title 1.2 安裝套件（約 3–5 分鐘，只需跑一次）
%%capture
import os
IS_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

if IS_COLAB:
    !pip install -q unsloth
    !pip install -q --upgrade --no-cache-dir --no-deps unsloth unsloth_zoo
    # 評測與資料處理
    !pip install -q "datasets>=3.2.0" "pandas>=2.3.0" pyarrow pyyaml datasketch matplotlib

In [ ]:
#@title 1.2b 驗收：每一項都要能 import（unsloth 除外——只驗安裝、不 import）
import importlib, importlib.util, importlib.metadata, sys

# 【坑】`import unsloth` 會立刻 patch 全域的 transformers。
#   §4.2 的「原生 HF baseline」必須在 unsloth 從未 import 過的 process 裡跑，
#   所以這一格對 unsloth 只檢查「裝好了沒」，不真的 import。
#   unsloth 的第一次 import 延後到 §4.3（HF baseline 之後）。
need_import  = ["torch", "transformers", "peft", "trl", "datasets", "bitsandbytes", "pandas", "pyarrow"]
need_present = ["unsloth"]
bad = []
for m in need_import:
    try:
        mod = importlib.import_module(m)
        print(f"  ok   {m:<14} {getattr(mod, '__version__', '?')}")
    except Exception as e:
        print(f"  FAIL {m:<14} {type(e).__name__}: {e}")
        bad.append(m)
for m in need_present:
    if importlib.util.find_spec(m) is None:
        print(f"  FAIL {m:<14} 未安裝")
        bad.append(m)
    else:
        try: v = importlib.metadata.version(m)
        except Exception: v = "?"
        print(f"  ok   {m:<14} {v}（只驗安裝，未 import）")
if "unsloth" in sys.modules:
    print("\n  ⚠️ 這個 session 已經 import 過 unsloth。若 §4.2 的 HF baseline 還沒有成功紀錄，")
    print("     它會在 §4.2 停下來要求重啟——照指示做即可。")
assert not bad, f"缺套件：{bad} —— 重跑 1.2，若仍失敗則「執行階段 → 重新啟動工作階段」後再跑一次"
print("\n全部就緒")



## 1.3 掛 Drive、決定路徑

**所有產出都寫進 Drive。**Colab 的本機磁碟在斷線後會清空，8 小時的實驗會歸零。

In [ ]:
#@title 1.3 掛 Drive 並建立目錄
import os, json, time
from pathlib import Path

USE_DRIVE = True  #@param {type:"boolean"}

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/ultrascale-lab-week3')
else:
    ROOT = Path('/content/ultrascale-lab-week3')

for sub in ['data', 'out', 'results', 'reports', 'datasets', 'logs']:
    (ROOT / sub).mkdir(parents=True, exist_ok=True)

# 大檔（模型權重、adapter）放本機碟，只把小的結果檔同步回 Drive。
# 原因：Drive 的 I/O 很慢，訓練時每步寫 checkpoint 會拖垮速度。
# 【坑：什麼該放 Drive、什麼可以放本機碟】
#   執行階段被回收時，/content 底下的東西**全部消失**，但 Drive 上的會留著。
#   原本我把 adapter 存在 /content/scratch，而完成紀錄（train_*.json）存在 Drive ——
#   結果執行階段一斷，紀錄還在所以訓練被跳過，adapter 卻已經沒了，
#   評測就會指到一個不存在的路徑。這是最糟的一種失敗：看起來一切正常。
#   → adapter 只有約 120 MB，一律存 Drive。只有「可以重新下載」的 HF 快取放本機碟。
SCRATCH = Path('/content/scratch')
for sub in ['out', 'hf_cache']:
    (SCRATCH / sub).mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(SCRATCH / 'hf_cache')

# ---- 完成紀錄的讀寫。定義在這裡而不是 §4，是因為 §1.5 的續跑狀態就要用；
#      Python 是由上而下執行的，helper 一定要比第一個使用者早。 ----
def done_path(tag):
    return ROOT / 'results' / f'{tag}.json'

def is_done(tag):
    return done_path(tag).exists()

def mark_done(tag, payload):
    done_path(tag).write_text(json.dumps(payload, ensure_ascii=False, indent=2))
    print(f"  → 已寫入 results/{tag}.json")

print("結果（要保留）:", ROOT)
print("暫存（可丟棄）:", SCRATCH)
print("完成紀錄     :", ROOT/'results')


## 1.4 HuggingFace 登入

`google/gemma-3-*` 需要接受授權條款。**兩個都要接受**（`-pt` 和 `-it` 是分開的 repo）：

- https://huggingface.co/google/gemma-3-4b-it
- https://huggingface.co/google/gemma-3-4b-pt

然後在 Colab 左側的「🔑 密鑰」加一個叫 `HF_TOKEN` 的密鑰（token 從 https://huggingface.co/settings/tokens 產生，read 權限即可），並把下一格的 `USE_OFFICIAL_REPO` 勾成 `True`。

**預設是 `False`，用 `unsloth/gemma-3-4b-*` 鏡像 —— 權重與官方相同、不需要接受授權條款。**沒有 token 的話直接跑就好。

（`USE_OFFICIAL_REPO=True` 但找不到 token 會直接報錯，不會安靜地退回鏡像 —— 否則你會以為在跑官方權重，其實不是。）

In [ ]:
#@title 1.4 登入並決定用哪個 repo
from huggingface_hub import login
import os

tok = None
try:
    from google.colab import userdata
    tok = userdata.get('HF_TOKEN')
except Exception:
    tok = os.environ.get('HF_TOKEN')

USE_OFFICIAL_REPO = False  #@param {type:"boolean"}
# False = unsloth 鏡像（權重相同、不需授權，預設）
# True  = google 官方 repo（需要 HF_TOKEN 且兩個 repo 的條款都已接受）

if tok:
    login(token=tok)
    print("已登入 HF。")
elif USE_OFFICIAL_REPO:
    raise RuntimeError("USE_OFFICIAL_REPO=True 但找不到 HF_TOKEN。"
                       "請在左側「🔑 密鑰」新增 HF_TOKEN，或把它改回 False。")

if USE_OFFICIAL_REPO:
    MODEL_IT, MODEL_PT = "google/gemma-3-4b-it", "google/gemma-3-4b-pt"
else:
    MODEL_IT, MODEL_PT = "unsloth/gemma-3-4b-it", "unsloth/gemma-3-4b-pt"
    print("使用 unsloth 鏡像（不需授權，權重與官方相同）。")

# 4-bit 量化版（下載快 4 倍，訓練時本來就會量化）
MODEL_IT_4BIT = MODEL_IT + "-bnb-4bit"
MODEL_PT_4BIT = MODEL_PT + "-bnb-4bit"

print(f"INSTRUCT : {MODEL_IT}")
print(f"BASE     : {MODEL_PT}   ← Shadow-FT 在這個上面訓練")

In [ ]:
#@title 1.5 續跑狀態：斷線之後先跑這一格，看還缺什麼
import json
from pathlib import Path

def sweep_status():
    rows = []
    for kind, tags in [
        ("框架對照", ["ab_hf_fp16", "ab_hf_fp32", "ab_unsloth"]),
        ("訓練",     [f"train_{t}" for t in
                      ("A1","A2","A3","A4","A5","B1","B3","B4","C1","D1","D2")]),
        ("快速評測", [f"eval_quick_{t}" for t in
                      ("base_it","A1","A2","A3","A4","A5","B1","B3","B4","C1","D1","D2")]),
        ("完整評測", [f"eval_full_{t}"  for t in ("base_it","A3","A5","D1","D2")]),
    ]:
        for t in tags:
            ok = is_done(t)
            note = ""
            if ok and t.startswith("train_"):
                adp = Path(json.loads(done_path(t).read_text()).get("adapter_dir", ""))
                if not (adp / "adapter_config.json").exists():
                    ok, note = False, "⚠️ 紀錄在但 adapter 不見了，會自動重訓"
            rows.append((kind, t, ok, note))
    return rows

rows = sweep_status()
cur = None
for kind, t, ok, note in rows:
    if kind != cur:
        print(f"\n【{kind}】"); cur = kind
    print(f"  {'✅' if ok else '  '} {t:<16}{note}")

done = sum(1 for *_, ok, _ in [(r[0],r[1],r[2],r[3]) for r in rows] if ok)
print(f"\n完成 {done}/{len(rows)}。")
print("\n【這一格只是報告，不會改變任何行為】")
print("  真正的跳過是每一格自己在做的：訓練與評測看 results/*.json、")
print("  資料看檔案在不在、σ 看 reports/shadow_ft_sigma.json。")
print("  所以直接從頭 Run All 就好，不需要手動跳過任何格子。")
print("  已完成的會印 [skip] 秒過；唯一每次都要重跑的是 §1.2 裝套件（3–5 分鐘）。")
print("大檔的位置：")
print(f"  adapter（要留） : {ROOT/'out'}  ← Drive，斷線不會消失")
print(f"  HF 模型快取     : {SCRATCH/'hf_cache'}  ← /content，斷線會消失但可重新下載")


---
# §2 資料

Week 2 用的是 `twinkle-ai/tw-reasoning-instruct-50k`，8,000 筆抽樣（train 7,600 / val 400），seed 42。這裡完全沿用**同一份抽樣**，只換渲染模板。

## 2.1 一個必須做的決定：`think` 欄位怎麼渲染

Gemma 4 有 thinking channel，Week 2 把 CoT 放進 `<|channel>thought`。**Gemma 3 沒有這個東西。**三個選項：

| 選項 | 做法 | 後果 |
|---|---|---|
| **`inline`（預設）** | 把 think 用 `<think>…</think>` 包起來，放在回答前面，都在 model turn 裡 | 最接近 Week 2；但也最容易複製 Week 2 的格式崩潰（模型學會「先長篇思考再散文回答」） |
| `drop` | 完全丟掉 think，只留最終回答 | 訓練訊號變短變乾淨，格式風險最低；但丟掉了資料集最有價值的部分 |
| `inline_mixed` | `inline` + 混入 5% 的 `\box{X}` 格式樣本（取自 TMMLU+ **訓練科目**，與評測的三科不重疊） | Week 2 總結第 4.4 節的修法 B。**推薦**：能同時測「格式崩潰」和「混入格式樣本能不能救」 |

**這一格會全部產生**，之後每個實驗指定要用哪一份。

In [ ]:
#@title 2.1 下載並渲染訓練資料
import json, random, re
from pathlib import Path
from datasets import load_dataset
from transformers import AutoTokenizer

N_SAMPLE   = 8000   #@param {type:"integer"}
VAL_RATIO  = 0.05   #@param {type:"number"}
SEED       = 42     #@param {type:"integer"}
MAX_CHARS  = 8000   #@param {type:"integer"}

DATA = ROOT / 'data'
tok_it = AutoTokenizer.from_pretrained(MODEL_IT)   # ← 一律用 INSTRUCT 的 tokenizer/模板

# ---- Gemma 3 的 chat template 支不支援 system role？先測，不要假設 ----
def supports_system(tk):
    try:
        tk.apply_chat_template(
            [{"role": "system", "content": "x"}, {"role": "user", "content": "y"}],
            tokenize=False, add_generation_prompt=True)
        return True
    except Exception:
        return False

HAS_SYSTEM = supports_system(tok_it)
print(f"Gemma 3 chat template 支援 system role: {HAS_SYSTEM}")

def render(user, assistant, system=None):
    # 回傳訓練用的純文字。注意結尾不含 generation prompt。
    msgs = []
    if system:
        if HAS_SYSTEM:
            msgs.append({"role": "system", "content": system})
        else:
            user = system + "\n\n" + user      # 退回：折進第一個 user turn
    msgs.append({"role": "user", "content": user})
    msgs.append({"role": "assistant", "content": assistant})
    text = tok_it.apply_chat_template(msgs, tokenize=False)
    # 【坑】apply_chat_template 會在最前面加 <bos>，而 SFTTrainer 之後 tokenize 時
    #      預設 add_special_tokens=True 又會再加一個 → 雙 BOS，訓練與推論的前綴不一致。
    #      這裡先剝掉，讓 tokenizer 統一負責。
    if tok_it.bos_token and text.startswith(tok_it.bos_token):
        text = text[len(tok_it.bos_token):]
    return text

if not (DATA / 'train_inline.jsonl').exists():
    raw = load_dataset("twinkle-ai/tw-reasoning-instruct-50k", split="train")
    print(f"step1 載入      : {len(raw):,}")

    def ok(r):
        if not (r.get('input') and r.get('output')): return False
        n = len(r.get('input','')) + len(r.get('think') or '') + len(r.get('output',''))
        return 20 <= n <= MAX_CHARS
    raw = raw.filter(ok)
    print(f"step2 清理      : {len(raw):,}")

    seen, keep = set(), []
    for i, r in enumerate(raw):
        k = r['input'].strip()
        if k in seen: continue
        seen.add(k); keep.append(i)
    raw = raw.select(keep)
    print(f"step3 精確去重  : {len(raw):,}")

    raw = raw.shuffle(seed=SEED).select(range(min(N_SAMPLE, len(raw))))
    n_val = int(len(raw) * VAL_RATIO)
    splits = {'valid': raw.select(range(n_val)), 'train': raw.select(range(n_val, len(raw)))}
    print(f"step4 抽樣切分  : train {len(splits['train']):,} / valid {len(splits['valid']):,}")

    for mode in ('inline', 'drop'):
        for name, ds in splits.items():
            out = DATA / f'{name}_{mode}.jsonl'
            with out.open('w') as f:
                for r in ds:
                    think = (r.get('think') or '').strip()
                    ans = r['output'].strip()
                    body = f"<think>\n{think}\n</think>\n\n{ans}" if (mode == 'inline' and think) else ans
                    f.write(json.dumps({"text": render(r['input'].strip(), body)}, ensure_ascii=False) + "\n")
            print(f"  寫出 {out.name:<22} {sum(1 for _ in out.open()):,} 筆")
else:
    print("資料已存在，跳過。要重做請刪掉 data/*.jsonl")

print()
print("=== 一筆完整樣本（前 700 字）===")
print(json.loads((DATA / 'train_inline.jsonl').open().readline())['text'][:700])


## 2.2 混入 `\box{}` 格式樣本（`inline_mixed`）

Week 2 總結 4.4 節的修法 B。**關鍵是科目不能重疊**：格式樣本只從 TMMLU+ 的**其他科目**抽，評測用的那三科（台灣地理、台語、三民主義）一題都不能碰，否則就是資料洩漏。

In [ ]:
#@title 2.2 產生 inline_mixed（混入 5% 格式樣本）
import pandas as pd, random, json
from datasets import load_dataset

MIX_RATIO = 0.05  #@param {type:"number"}
EVAL_SUBJECTS = ["geography_of_taiwan", "taiwanese_hokkien", "three_principles_of_people"]

SYS_BOX = (
    "使用者將提供一個題目，並附上選項 A、B、C、D。\n"
    "請仔細閱讀題目要求，根據題意選出最符合的選項，並將選項以以下格式輸出：\n"
    "\\box{選項}\n"
    "請確保僅將選項包含在 { } 中，否則將不計算為有效答案。\n"
    "務必精確遵循輸出格式，避免任何多餘內容或錯誤格式。\n"
    "例如：答案是 A，就輸出 \\box{A}。\n"
)

TMMLU_DIR = ROOT / 'datasets' / 'tmmluplus'
TMMLU_DIR.mkdir(parents=True, exist_ok=True)

# ---- 下載 TMMLU+ ----
if not any(TMMLU_DIR.glob("*.parquet")):
    from huggingface_hub import snapshot_download
    p = snapshot_download(repo_id="ikala/tmmluplus", repo_type="dataset",
                          local_dir=str(TMMLU_DIR / '_raw'))
    import shutil
    for f in Path(p).rglob("*test*.csv"):
        pd.read_csv(f).to_parquet(TMMLU_DIR / (f.stem.replace('_test','') + '.parquet'))
    for f in Path(p).rglob("*.parquet"):
        if f.parent != TMMLU_DIR:
            shutil.copy(f, TMMLU_DIR / f.name)
subjects = sorted(f.stem for f in TMMLU_DIR.glob("*.parquet"))
print(f"TMMLU+ 科目數：{len(subjects)}")
assert len(subjects) >= 10, "TMMLU+ 下載失敗，檢查上一步"

# ---- 抽格式樣本（排除評測科目 + 題目層級排除跨科目重複）----
# 【坑】TMMLU+ 有題目在不同科目之間重複出現。只用「科目名稱」排除不夠——
#   別科抽到的題目可能和評測三科的某題一字不差。所以再用「題目全文」排除一次。
_ban = set()
for _s in EVAL_SUBJECTS:
    _dfe = pd.read_parquet(TMMLU_DIR / f"{_s}.parquet")
    _ban |= set(_dfe['question'].astype(str).str.strip())

def build_mixed():
    pool = [s for s in subjects if s not in EVAL_SUBJECTS]
    rng = random.Random(SEED)
    n_train = sum(1 for _ in (DATA / 'train_inline.jsonl').open())
    n_mix = int(n_train * MIX_RATIO / (1 - MIX_RATIO))

    rows = []
    for s in pool:
        df = pd.read_parquet(TMMLU_DIR / f"{s}.parquet")
        for _, r in df.iterrows():
            if all(k in r and pd.notna(r[k]) for k in ("question","A","B","C","D","answer")) \
               and str(r['question']).strip() not in _ban:
                rows.append((s, r))
    rng.shuffle(rows)
    rows = rows[:n_mix]

    with (DATA / 'train_inline_mixed.jsonl').open('w') as f:
        for line in (DATA / 'train_inline.jsonl').open():
            f.write(line)
        for s, r in rows:
            q = r['question'] + "\n" + "\n".join(f"{k}: {r[k]}" for k in "ABCD")
            f.write(json.dumps({"text": render(q, f"\\box{{{str(r['answer']).strip().upper()}}}",
                                               system=SYS_BOX)}, ensure_ascii=False) + "\n")
    # 留一份「混了哪些題」的清單，之後查資料問題不用反解 jsonl
    json.dump([{"subject": s, "question": str(r['question'])} for s, r in rows],
              (DATA / 'train_inline_mixed.sources.json').open('w'), ensure_ascii=False, indent=1)
    import shutil; shutil.copy(DATA/'valid_inline.jsonl', DATA/'valid_inline_mixed.jsonl')
    print(f"混入 {len(rows):,} 筆格式樣本（{100*len(rows)/(n_train+len(rows)):.1f}%），"
          f"來自 {len(pool)} 個非評測科目（已另排除 {len(_ban)} 個評測題目全文）")

if not (DATA / 'train_inline_mixed.jsonl').exists():
    build_mixed()
else:
    print("inline_mixed 已存在，先做完整洩漏檢查再決定要不要重生成。")

# ---- 防呆：評測三科的題目絕不能出現在訓練資料裡 ----
# 【坑】第一版檢查有兩個 bug，造成「上次過、這次炸」的抽籤式結果：
#   1. list(set(...))[:200] —— set 的順序每個 process 都不同（字串雜湊隨機化），
#      每個 session 抽到的 200 題不一樣，檢查等於抽籤。
#   2. 只比題幹前 40 字、對 raw JSON 行 —— 短而普通的題幹會撞到
#      別科題目或基底語料，出現假陽性；含引號的題目被跳脫後又比不到（假陰性）。
#   改成：全部題目、題目全文、比對 json 解碼後的 text，並區分洩漏來源：
#   - 出現在「混入的格式樣本」→ TMMLU+ 跨科目重複 → 自動用題目層級排除重生成
#   - 出現在「基底語料 train_inline.jsonl」→ 影響所有 *_inline 的 run，硬停，人工處理
def leakage_report():
    n_base = sum(1 for _ in (DATA / 'train_inline.jsonl').open())
    base_texts, mix_texts = [], []
    for i, l in enumerate((DATA / 'train_inline_mixed.jsonl').open()):
        (base_texts if i < n_base else mix_texts).append(json.loads(l)["text"])
    base_blob = "\n\x00\n".join(base_texts)   # \x00 當分隔，避免跨行拼出假匹配
    mix_blob  = "\n\x00\n".join(mix_texts)
    hits_base, hits_mix, skipped = [], [], 0
    for s in EVAL_SUBJECTS:
        df = pd.read_parquet(TMMLU_DIR / f"{s}.parquet")
        for q in df['question'].astype(str):
            qq = q.strip()
            if len(qq) < 15:
                skipped += 1; continue   # 太短的題幹到處都撞得到，不足以當洩漏證據
            if qq in mix_blob:  hits_mix.append((s, qq))
            if qq in base_blob: hits_base.append((s, qq))
    if skipped:
        print(f"（{skipped} 題題幹短於 15 字，略過不作為證據）")
    return hits_base, hits_mix

hits_base, hits_mix = leakage_report()

if hits_mix:
    print(f"⚠️ 混入的格式樣本含 {len(hits_mix)} 筆評測題目（TMMLU+ 跨科目重複），例如：")
    for s, q in hits_mix[:5]:
        print(f"   [{s}] {q[:60]}")
    print("→ 自動重新生成 inline_mixed（這批樣本沒被任何已完成的訓練用過，重生成無副作用）…")
    (DATA / 'train_inline_mixed.jsonl').unlink()
    build_mixed()
    hits_base, hits_mix = leakage_report()

assert not hits_mix, f"重生成後混入樣本仍有洩漏（不應發生，回報這個 bug）：{hits_mix[:3]}"
assert not hits_base, (
    f"❌ 基底語料 train_inline.jsonl 本身含 {len(hits_base)} 筆評測題目，例如：\n"
    + "\n".join(f"   [{s}] {q[:60]}" for s, q in hits_base[:5]) +
    "\n  這不是 inline_mixed 的問題——所有用 *_inline 訓練的 run 都受影響。"
    "\n  不要只重生成 inline_mixed，要回頭查 Week 2 抽樣語料的來源再決定怎麼辦。")
print("防呆通過：評測三科的題目沒有出現在訓練資料中（全部題目、全文比對）")



---
# §3 記憶體預測（先算再量）

沿用 Week 1／Week 2 的做法：**先用公式算出預測值，再去量，然後解釋差在哪。**Week 2 最大的收穫之一就是「公式方向對，但直接代入會系統性高估」。

Gemma 3 4B 這裡有一個和 Week 2 一模一樣的陷阱：**262,144 的 vocab**。logits 是 `seq × bs × V × (2 + 4) bytes`，seq=1024、bs=2 時就是 **3.00 GiB**（= 3.22 GB，注意單位）—— 比 4-bit 權重還大。Week 2 的 H3 就是被這一項稀釋掉梯度檢查點的效益。

**Unsloth 的 fused / chunked cross-entropy 正好是針對這一項。**所以 §4 的框架對照裡，這一項應該是 HF 和 Unsloth 差最多的地方 —— 這是一個事先就能寫下來的預測。

In [ ]:
#@title 3.1 從 config.json 推算，不要用記憶中的規格
import json, math
from huggingface_hub import hf_hub_download

cfg = json.load(open(hf_hub_download(MODEL_IT, "config.json")))
tc = cfg.get("text_config", cfg)

H   = tc["hidden_size"]; L = tc["num_hidden_layers"]
NH  = tc["num_attention_heads"]; NKV = tc.get("num_key_value_heads", NH)
HD  = tc.get("head_dim", H // NH); FF = tc["intermediate_size"]
V   = tc.get("vocab_size", cfg.get("vocab_size"))

print(f"hidden_size {H} | layers {L} | heads {NH} | kv_heads {NKV} | head_dim {HD}")
print(f"intermediate {FF} | vocab {V:,}")

emb   = V * H
attn  = H*NH*HD + 2*(H*NKV*HD) + NH*HD*H
mlp   = 3 * H * FF
per_l = attn + mlp
text  = emb + L*per_l
print(f"\n語言主幹參數：{text/1e9:.3f}B（embed {emb/1e9:.3f}B + {L}×{per_l/1e6:.1f}M）")

# ⚠️ 這個常數不能從 Week 2 直接搬。
# MLX 的 4-bit 是 group_size 64 + bf16 的 scale/bias → 4 + 32/64 = 4.50 bit/param
# （out/gemma4-e4b-tw/model.safetensors 實測 4.501，用它預測誤差是 -0.0%）。
# 這裡用的是 bitsandbytes 的 nf4 + double quant，機制不同（block 64、absmax 再量化一次），
# 名目上也是約 4.5 bit/param，但**要用 §4 量到的峰值回頭校正**，不要當成已知。
BPP_4BIT = 4.5/8
print(f"4-bit 權重     ≈ {text*BPP_4BIT/2**30:.2f} GiB")

def lora_params(rank, attn_only):
    q = H*rank + rank*NH*HD; k = H*rank + rank*NKV*HD
    o = NH*HD*rank + rank*H
    a = q + 2*k + o
    m = 2*(H*rank + rank*FF) + (FF*rank + rank*H)
    return L * (a if attn_only else a + m)

print()
print(f"{'設定':<28}{'可訓練參數':>14}{'Adam 狀態':>14}")
for rank in (8, 16, 32, 64):
    for ao, lbl in ((True,'attn-only'), (False,'all-linear')):
        p = lora_params(rank, ao)
        print(f"  r={rank:<3} {lbl:<18}{p/1e6:>12.2f}M{p*16/2**30:>12.2f} GiB")

print()
print(f"{'seq × bs':<14}{'logits 記憶體':>16}")
for seq in (512, 1024, 2048):
    for bs in (1, 2, 4):
        print(f"  {seq}×{bs:<8}{seq*bs*V*6/2**30:>14.2f} GiB")
print("\n→ 這一項不受梯度檢查點影響（Week 2 H3 實測）。")
print("→ Unsloth 的 fused CE 不會把完整 logits 落地，§4 應該看得到差別。")

json.dump({"hidden": H, "layers": L, "vocab": V, "text_params": text,
           "pred_weights_4bit_gib": text*BPP_4BIT/2**30},
          open(ROOT/'reports'/'memory_prediction_gemma3.json','w'), indent=2)


## 3.2 驗證 Shadow-FT 的前提：base 和 instruct 的權重有多接近？

論文定義相對差距 **σ = Σ|W_B − W_I| / (Σ|W_B| + Σ|W_I|)**，並宣稱所有測過的模型 σ < 0.05。這是整個方法能成立的基礎——如果我們這一對的 σ 很大，Shadow-FT 就不該預期有效。

**這一格是「先驗證前提再做實驗」，不是可有可無的。**用 `safe_open` 逐張量比對，記憶體佔用是常數，不會 OOM。

In [ ]:
#@title 3.2 計算 σ（Shadow-FT Eq.1）—— 串流分塊版，RAM 固定
import torch, json, gc
from huggingface_hub import snapshot_download
from safetensors import safe_open
from pathlib import Path

RUN_SIGMA   = True    #@param {type:"boolean"}
FORCE_SIGMA = False   #@param {type:"boolean"}
ADD_LM_ONLY = True    #@param {type:"boolean"}
CHUNK_ROWS  = 4096    #@param {type:"integer"}

# 算過就自動跳過（結果在 Drive，斷線也還在）。要重算就把 FORCE_SIGMA 打開。
_SIGMA_OUT = ROOT / 'reports' / 'shadow_ft_sigma.json'
if RUN_SIGMA and _SIGMA_OUT.exists() and not FORCE_SIGMA:
    _prev = json.loads(_SIGMA_OUT.read_text())
    if 'sigma_lm_only' in _prev or not ADD_LM_ONLY:
        print(f"[skip] σ 已經算過：全部 {_prev['sigma']:.4f}"
              f"｜僅 language_model {_prev.get('sigma_lm_only', '—')}"
              f"（{_prev['n_tensors']} 個張量）→ {_SIGMA_OUT}")
        print("       要重算就把 FORCE_SIGMA 打開。")
        RUN_SIGMA = False
    else:
        # 舊紀錄只有「全部張量」的 σ，而差最大的張量是 multi_modal_projector——
        # 論文的 σ 是對 LLM 權重算的，vision/projector 會灌水。重算一次補上
        # language_model-only 的數字（只吃磁碟與時間 ~30 分鐘，不吃 GPU 配額）。
        # 不想等就把 ADD_LM_ONLY 關掉。
        print(f"舊紀錄缺 sigma_lm_only，重算補上（ADD_LM_ONLY=False 可跳過）…")

# 【坑：這一格是最容易把 Colab 免費版的 RAM 打爆的地方】
# 直覺寫法是 h.get_tensor(k).float()，但 embed_tokens 是 262,144 × 2,560：
#   fp16 1.34 GB → .float() 變 2.68 GB，兩個模型各一份就 5.4 GB，
#   再加 (a-b) 的中間張量 → 8 GB 以上。免費版總共只有 ~12.7 GB，kernel 直接重啟。
# 正解：safe_open 的 get_slice() 是惰性的，可以只讀某幾列。
#       逐塊累加成純量，RAM 佔用和張量大小無關。

if RUN_SIGMA:
    print_ram("開始前")

    def index(repo):
        d = Path(snapshot_download(repo, allow_patterns=["*.safetensors", "*.json"]))
        m = {}
        for f in sorted(d.glob("*.safetensors")):
            with safe_open(f, framework="pt") as h:
                for k in h.keys():
                    m[k] = f
        return m

    print("下載 BASE 與 INSTRUCT 的 fp16 權重（各約 8.6 GB，只佔磁碟不佔 RAM）…")
    ib, ii = index(MODEL_PT), index(MODEL_IT)
    common = sorted(set(ib) & set(ii))
    print(f"BASE {len(ib)} 張量 / INSTRUCT {len(ii)} 張量 / 共同 {len(common)}")
    only_b, only_i = sorted(set(ib) - set(ii)), sorted(set(ii) - set(ib))
    if only_b or only_i:
        print(f"  ⚠️ 只在 BASE: {only_b[:3]} … 只在 IT: {only_i[:3]} …")

    def sums(fb, fi, k, chunk):
        # 回傳 (Σ|W_B − W_I|, Σ|W_B| + Σ|W_I|)，逐塊累加，峰值 RAM = 一塊的大小
        with safe_open(fb, framework="pt") as hb, safe_open(fi, framework="pt") as hi:
            sb, si = hb.get_slice(k), hi.get_slice(k)
            shape_b, shape_i = sb.get_shape(), si.get_shape()
            if shape_b != shape_i:
                return None, None, (shape_b, shape_i)
            n = d = 0.0
            if not shape_b:                       # 純量
                a = hb.get_tensor(k).double(); b = hi.get_tensor(k).double()
                return (a - b).abs().sum().item(), a.abs().sum().item() + b.abs().sum().item(), None
            for st in range(0, shape_b[0], chunk):
                en = min(st + chunk, shape_b[0])
                a = sb[st:en].double()
                b = si[st:en].double()
                n += (a - b).abs().sum().item()
                d += a.abs().sum().item() + b.abs().sum().item()
                del a, b
            return n, d, None

    num = den = 0.0
    num_lm = den_lm = 0.0
    per_tensor, mismatch = {}, []
    for j, k in enumerate(common):
        n, d, bad = sums(ib[k], ii[k], k, CHUNK_ROWS)
        if bad:
            mismatch.append((k, bad)); continue
        num += n; den += d
        if k.startswith("language_model."):
            num_lm += n; den_lm += d
        per_tensor[k] = n / d if d else 0.0
        if j % 50 == 0:
            gc.collect(); print_ram(f"{j}/{len(common)}")

    assert not mismatch, f"形狀不符，兩個模型不是同一架構，不能做 Shadow-FT：{mismatch[:3]}"

    sigma = num / den
    sigma_lm = (num_lm / den_lm) if den_lm else None
    n_lm = sum(1 for k in per_tensor if k.startswith("language_model."))
    print(f"\nσ(BASE, INSTRUCT) 全部張量           = {sigma:.4f}")
    if sigma_lm is not None:
        print(f"σ(BASE, INSTRUCT) 僅 language_model = {sigma_lm:.4f}（{n_lm} 個張量）")
    # 判定用 language_model-only：論文的 σ 是對 LLM 權重算的，
    # vision tower / multi_modal_projector 不在 Shadow-FT 要搬的 delta 裡。
    _verdict = sigma_lm if sigma_lm is not None else sigma
    print("論文門檻 < 0.05 → "
          + ("✅ 前提成立" if _verdict < 0.05 else
             "⚠️ 偏大，Shadow-FT 的前提不成立——D1/D2 的結果要當作「前提外推」來寫，"
             "手冊 §6.2 說這時要先停下來想為什麼"))

    top = sorted(per_tensor.items(), key=lambda x: -x[1])[:8]
    print("\n差最多的 8 個張量：")
    for k, v in top: print(f"  {v:.4f}  {k}")

    top_lm = sorted(((k, v) for k, v in per_tensor.items()
                     if k.startswith("language_model.")), key=lambda x: -x[1])[:8]
    json.dump({"sigma": sigma, "sigma_lm_only": sigma_lm,
               "n_tensors": len(common), "n_tensors_lm": n_lm, "chunk_rows": CHUNK_ROWS,
               "top_divergent": [{"name": k, "sigma": v} for k, v in top],
               "top_divergent_lm": [{"name": k, "sigma": v} for k, v in top_lm]},
              open(ROOT / 'reports' / 'shadow_ft_sigma.json', 'w'), indent=2)
    json.dump(per_tensor, open(ROOT / 'reports' / 'shadow_ft_sigma_per_tensor.json', 'w'), indent=2)
    free_ram(ib, ii, per_tensor)
    print_ram("結束後")


### 跑完這一格可以把 fp16 權重刪掉

σ 算完之後就用不到 fp16 權重了（訓練與評測都用 4-bit）。留著會佔約 17 GB 磁碟。

In [ ]:
#@title 3.3 （選用）刪掉只為了算 σ 而下載的 fp16 權重
import shutil, os
from pathlib import Path

DELETE_FP16 = False  #@param {type:"boolean"}

cache = Path(os.environ.get('HF_HOME', Path.home()/'.cache'/'huggingface')) / 'hub'
targets = [d for d in cache.glob('models--*')
           if 'gemma-3-4b' in d.name and 'bnb' not in d.name and '4bit' not in d.name]
for d in targets:
    sz = sum(f.stat().st_size for f in d.rglob('*') if f.is_file()) / 2**30
    print(f"  {d.name}  {sz:.1f} GiB" + ("  → 刪除" if DELETE_FP16 else "  （DELETE_FP16=False，保留）"))
    if DELETE_FP16:
        shutil.rmtree(d)
if not targets:
    print("  沒有找到 fp16 快取。")


---
# §4 框架對照：HF transformers+peft vs Unsloth

**這一節是為了回答主管的 Q1。**同一份資料、同一組 LoRA 參數、同樣 30 步，量三件事：峰值記憶體、每步耗時、loss 曲線。

**事先預測**（寫在跑之前，跑完再回來對）：

| 項目 | 預測 | 理由 |
|---|---|---|
| HF + fp16 | **會 NaN** | Gemma 3 layernorm 後 activation 超過 fp16 上限 65,504（廠商說法，本格就是在驗證它） |
| HF + fp32 | 能跑，但**慢 3–5 倍** | T4 fp32 只有 8.1 TFLOPS，fp16 有 65 TFLOPS |
| Unsloth | 能跑，且峰值記憶體**明顯較低** | fused CE 不落地完整 logits（§3.1 算出來是 3.00 GiB） |
| loss | 三者**應該接近** | 若差很多，代表某一邊的數學不對，不是優化 |

最後一列是最重要的對帳：**優化框架的正當性建立在「數值等價」上**。如果 Unsloth 的 loss 曲線和 HF 對不起來，那省下來的記憶體就不能算數。

In [ ]:
#@title 4.1 共用工具：記憶體與計時
import torch, time, json, gc, os
from pathlib import Path

def reset_mem():
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

def peak_gib():
    return torch.cuda.max_memory_allocated() / 2**30

def baseline_gib():
    # reset_mem() 之後仍佔著的 VRAM ＝ 同一個 session 之前殘留、放不掉的部分。
    # peak_gib 會把它整段算進去，所以要引用記憶體數字時用 peak - baseline。
    return torch.cuda.memory_allocated() / 2**30

def require_stage_ready(stage, pending, need_free=9.0, fresh_required=False):
    # 【為什麼在這裡擋】每跑完一組訓練會殘留 ~1.3 GiB 放不掉的 VRAM
    #   （free_all 的註解解釋了原因）。殘留讓 peak_gib 失真、累積幾組之後 OOM。
    #   Week3 第一輪實測：A1→A5 的 peak 每組整整多 1.259 GiB，就是這個。
    #   已完成的組會自動 [skip]，所以重啟後直接再按「全部執行」，
    #   幾分鐘內就會回到這裡繼續跑。
    if not pending:
        print(f"[skip] {stage}：全部完成")
        return
    gc.collect(); torch.cuda.empty_cache()
    alloc = baseline_gib()
    free, _ = vram()
    problems = []
    if fresh_required and alloc > 1.5:
        problems.append(f"這個 session 已殘留 {alloc:.1f} GiB（此階段要求乾淨 session）")
    if free < need_free:
        problems.append(f"可用 VRAM 只剩 {free:.1f} GiB（需要 ≥ {need_free:.1f}）")
    if problems:
        raise RuntimeError(
            f"⚠️ {stage} 還有 {pending} 沒跑，但{'；'.join(problems)}。\n"
            f"  請「執行階段 → 重新啟動工作階段」，然後直接再按一次「全部執行」。\n"
            f"  已完成的部分會自動 [skip]，不會重跑。")
    print(f"{stage}：待跑 {pending}｜VRAM 殘留 {alloc:.2f} GiB、可用 {free:.2f} GiB → 繼續")

def free_all(*objs):
    # 【坑】這個函式**放不掉呼叫端的變數**。objs 是新的區域參考，
    #   `del o` 只刪掉這裡的迴圈變數，呼叫端的 model / tr 還在。
    #   真正要釋放，必須在**持有變數的那個 scope** 裡 del。
    #   這裡只能做 gc + empty_cache。
    for o in objs:
        try: del o
        except Exception: pass
    gc.collect()
    try: torch.cuda.empty_cache()
    except Exception: pass

def vram():
    free, total = torch.cuda.mem_get_info()
    return free / 2**30, total / 2**30

def print_vram(tag=""):
    f, t = vram()
    used = t - f
    bar = "█" * int(20*used/t) + "·" * (20 - int(20*used/t))
    print(f"  VRAM {bar} 已用 {used:.2f}/{t:.2f} GiB（可用 {f:.2f}）{('  '+tag) if tag else ''}")

def require_vram(need_gib, what=""):
    # 重試之前一定要確認記憶體真的放掉了。沒有這道檢查，bitsandbytes 會
    # 「聰明地」把塞不下的部分丟到 CPU，然後拋一個看起來完全無關的
    # ValueError: Some modules are dispatched on the CPU or the disk。
    f, t = vram()
    if f < need_gib:
        raise RuntimeError(
            f"可用 VRAM 只有 {f:.2f} GiB，{what} 需要約 {need_gib:.1f} GiB。\n"
            f"  前一次的模型沒有被釋放乾淨。最可靠的解法是"
            f"「執行階段 → 重新啟動工作階段」再從頭 Run All（已完成的會 [skip]）。")

# done_path / is_done / mark_done 已在 §1.3 定義（§1.5 的續跑狀態要用，
# 必須比這裡早），這裡不要重複定義。

class StepTimer:
    # 用 TrainerCallback 記錄每步耗時與 loss，取後半平均（前幾步含 warmup 與編譯）。
    def __init__(self): self.t=[]; self.loss=[]; self._last=None
    def make(self):
        from transformers import TrainerCallback
        outer = self
        class CB(TrainerCallback):
            def on_step_begin(self, args, state, control, **kw): outer._last = time.time()
            def on_step_end(self, args, state, control, **kw):
                if outer._last: outer.t.append(time.time() - outer._last)
            def on_log(self, args, state, control, logs=None, **kw):
                if logs and 'loss' in logs: outer.loss.append(logs['loss'])
        return CB()
    def summary(self):
        h = self.t[len(self.t)//2:] or self.t
        return {"s_per_step": sum(h)/len(h) if h else None,
                "n_steps": len(self.t),
                "loss_first": self.loss[0] if self.loss else None,
                "loss_last": self.loss[-1] if self.loss else None,
                "loss_curve": self.loss}
print("ok")

In [ ]:
#@title 4.1b 版本相容層（TRL / Unsloth 的 API 這一年改過名，先探測再用）
import inspect

def make_sft_config(**kw):
    # TRL 新版用 max_length，舊版用 max_seq_length；dataset_text_field 也曾搬家。
    from trl import SFTConfig
    sig = set(inspect.signature(SFTConfig.__init__).parameters)
    if 'max_length' not in sig and 'max_seq_length' in sig and 'max_length' in kw:
        kw['max_seq_length'] = kw.pop('max_length')
    if 'max_seq_length' not in sig and 'max_length' in sig and 'max_seq_length' in kw:
        kw['max_length'] = kw.pop('max_seq_length')
    dropped = {k: kw.pop(k) for k in list(kw) if k not in sig}
    if dropped:
        print(f"  [compat] SFTConfig 不認得，已忽略：{list(dropped)}")
    return SFTConfig(**kw)

def for_inference(model):
    from unsloth import FastModel
    for fn in ('for_inference',):
        f = getattr(FastModel, fn, None)
        if f:
            try:
                return f(model)
            except Exception as e:
                print(f"  [compat] FastModel.{fn} 失敗（{e}），改用 model.eval()")
    model.eval()
    return model

print("相容層就緒")

In [ ]:
#@title 4.2 HF transformers + peft baseline（fp16 → 預期 NaN；再跑 fp32）
import torch, json, math, sys, gc
from datasets import load_dataset

N_STEPS_AB = 30   #@param {type:"integer"}
SEQ_AB     = 1024 #@param {type:"integer"}
BS_AB      = 1    #@param {type:"integer"}
# bs=1 是為了和 §5 的訓練保持同一個配置。bs=2 × seq=1024 的 logits 含 CE 副本
# 就要 3.00 GiB，T4 上很容易 OOM（見 §8.5）。三個框架用同一組值才比得起來，
# 實際用的值會記進 results/ab_*.json，之後看表不會弄混。

# 存過「error」的紀錄視同未完成 → 自動重試；成功的紀錄照舊 [skip] 不重跑。
# （Week3 第一輪的 fp16/fp32 都因為 unsloth 污染 + token_type_ids 掛掉，就是靠這裡重試。）
_ab_pending = []
for _dt in ("fp16", "fp32"):
    _t = f"ab_hf_{_dt}"
    if is_done(_t) and "error" not in json.loads(done_path(_t).read_text()):
        continue
    _ab_pending.append(_dt)

# 【坑】unsloth import 過一次就會 patch 全域 transformers，之後跑的就不是「原生 HF」。
#   Week3 第一輪兩組 HF 都拋 token_type_ids is required，原因之一就是 §1.2b 先
#   import 了 unsloth。§1.2b 已改成不 import unsloth，所以從頭 Run All 不會踩到；
#   只有先跑過 §4.3/§5 再回頭跑這格才會被擋。
if _ab_pending and "unsloth" in sys.modules:
    raise RuntimeError(
        f"⚠️ HF baseline 還缺 {_ab_pending}，但這個 session 已經 import 過 unsloth。\n"
        "  請「執行階段 → 重新啟動工作階段」，然後直接再按一次「全部執行」：\n"
        "  已完成的部分會自動 [skip]，Run All 會在 unsloth 第一次 import（§4.3）之前\n"
        "  先到這裡用原生 transformers 把 HF baseline 跑完。")

def run_hf(dtype_name):
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model
    from trl import SFTTrainer

    reset_mem()
    base0 = baseline_gib()
    dtype = {"fp16": torch.float16, "fp32": torch.float32}[dtype_name]
    m = tr = None
    try:
        tk = AutoTokenizer.from_pretrained(MODEL_IT)
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                 bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=dtype)
        m = AutoModelForCausalLM.from_pretrained(MODEL_IT, quantization_config=bnb,
                                                 dtype=dtype, device_map={"": 0},
                                                 attn_implementation="eager")
        m.config.use_cache = False
        m.gradient_checkpointing_enable()

        # 【坑】gemma-3-4b 的 checkpoint 是多模態包裝，訓練時 forward 可能硬性要求
        #   token_type_ids（0=文字、1=影像），但 SFTTrainer 的 collator 不會生它。
        #   純文字訓練補全 0 即可，數值上沒有任何影響。
        #   （就算某個 transformers 版本不要求，多傳一個全 0 也無害。）
        _fwd = m.forward
        def _fwd_tt(*a, **kw):
            ii = kw.get("input_ids")
            if kw.get("token_type_ids") is None and ii is not None:
                kw["token_type_ids"] = torch.zeros_like(ii)
            return _fwd(*a, **kw)
        m.forward = _fwd_tt

        m = get_peft_model(m, LoraConfig(
            r=16, lora_alpha=32, lora_dropout=0.0, bias="none", task_type="CAUSAL_LM",
            target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]))

        ds = load_dataset("json", data_files=str(ROOT/'data'/'train_inline.jsonl'), split="train")
        timer = StepTimer()
        tr = SFTTrainer(
            model=m, train_dataset=ds, processing_class=tk,
            args=make_sft_config(output_dir=str(SCRATCH/'out'/f'hf_{dtype_name}'),
                           per_device_train_batch_size=BS_AB, gradient_accumulation_steps=1,
                           max_steps=N_STEPS_AB, learning_rate=1e-4, logging_steps=1,
                           max_length=SEQ_AB, optim="adamw_torch",
                           fp16=(dtype_name=="fp16"), bf16=False,
                           gradient_checkpointing=True, report_to="none", seed=42,
                           save_strategy="no", dataset_text_field="text"),
            callbacks=[timer.make()])
        tr.train()
        pk = peak_gib()
        r = timer.summary()
        r.update({"framework": f"HF+peft ({dtype_name})", "peak_gib": pk,
                  "baseline_gib": base0, "peak_delta_gib": pk - base0})
        r["nan"] = bool(r["loss_last"] is None or (isinstance(r["loss_last"], float)
                        and (math.isnan(r["loss_last"]) or r["loss_last"] == 0.0)))
        return r
    finally:
        # 【坑】失敗時例外的 traceback 會抓著這個 frame → frame 抓著 model → VRAM 不放。
        #   所以 try/finally 在「持有變數的 scope」裡 del，成功失敗都收乾淨，
        #   fp16 掛掉才不會把殘骸留給 fp32 那一輪。
        try: del tr
        except Exception: pass
        try: del m
        except Exception: pass
        gc.collect()
        try: torch.cuda.empty_cache()
        except Exception: pass

AB = {}
for dt in ("fp16", "fp32"):
    tag = f"ab_hf_{dt}"
    if dt not in _ab_pending:
        AB[dt] = json.loads(done_path(tag).read_text()); print(f"[skip] {tag} 已成功過"); continue
    print(f"\n===== HF + {dt} =====")
    try:
        AB[dt] = run_hf(dt)
    except Exception as e:
        AB[dt] = {"framework": f"HF+peft ({dt})", "error": f"{type(e).__name__}: {e}"}
        print(f"  失敗：{AB[dt]['error']}")
    mark_done(tag, AB[dt])
    print(json.dumps({k: v for k, v in AB[dt].items() if k != 'loss_curve'},
                     ensure_ascii=False, indent=2))


In [ ]:
#@title 4.3 Unsloth（同樣參數、同樣 30 步）
#  ⚠️ 這一格跑完請「執行階段 → 重新啟動工作階段」再跑 §5。
#     unsloth 會 patch transformers，和上一格的 HF 混在同一個 process 裡容易出怪事，
#     而且訓練殘留的 VRAM 會把 §5 的 peak_gib 灌水。
#     忘記重啟也沒關係：§5.2 開頭的 require_stage_ready 會直接擋下來提醒你。
import json, math

def run_unsloth():
    from unsloth import FastModel
    from trl import SFTTrainer
    from datasets import load_dataset
    reset_mem()
    base0 = baseline_gib()
    model, tk = FastModel.from_pretrained(
        model_name=MODEL_IT, max_seq_length=SEQ_AB, load_in_4bit=True, full_finetuning=False)
    model = FastModel.get_peft_model(
        model, r=16, lora_alpha=32, lora_dropout=0.0, bias="none",
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        use_gradient_checkpointing="unsloth", random_state=42,
        finetune_vision_layers=False, finetune_language_layers=True)
    ds = load_dataset("json", data_files=str(ROOT/'data'/'train_inline.jsonl'), split="train")
    timer = StepTimer()
    tr = SFTTrainer(model=model, train_dataset=ds, processing_class=tk,
        args=make_sft_config(output_dir=str(SCRATCH/'out'/'unsloth_ab'),
                       per_device_train_batch_size=BS_AB, gradient_accumulation_steps=1,
                       max_steps=N_STEPS_AB, learning_rate=1e-4, logging_steps=1,
                       max_length=SEQ_AB, optim="adamw_8bit",
                       fp16=False, bf16=False,  # unsloth 自己決定精度
                       report_to="none", seed=42, save_strategy="no",
                       dataset_text_field="text"),
        callbacks=[timer.make()])
    tr.train()
    pk = peak_gib()
    r = timer.summary(); r.update({"framework": "Unsloth", "peak_gib": pk,
                                   "baseline_gib": base0, "peak_delta_gib": pk - base0})
    r["nan"] = bool(r["loss_last"] is None or (isinstance(r["loss_last"], float) and math.isnan(r["loss_last"])))
    free_all(model, tr)
    return r

tag = "ab_unsloth"
if is_done(tag):
    AB["unsloth"] = json.loads(done_path(tag).read_text()); print("已完成，跳過")
else:
    AB["unsloth"] = run_unsloth(); mark_done(tag, AB["unsloth"])
print(json.dumps({k:v for k,v in AB["unsloth"].items() if k!='loss_curve'}, ensure_ascii=False, indent=2))

In [ ]:
#@title 4.4 框架對照表（貼進報告用）
import json
rows = []
for tag, label in [("ab_hf_fp16","HF+peft fp16"), ("ab_hf_fp32","HF+peft fp32"), ("ab_unsloth","Unsloth")]:
    if is_done(tag):
        rows.append((label, json.loads(done_path(tag).read_text())))

print(f"{'框架':<18}{'峰值記憶體':>12}{'s/step':>10}{'相對速度':>10}{'首 loss':>10}{'末 loss':>10}  備註")
base_s = next((r[1].get('s_per_step') for r in rows if r[0]=='HF+peft fp32' and r[1].get('s_per_step')), None)
for label, r in rows:
    if 'error' in r:
        print(f"{label:<18}{'—':>12}{'—':>10}{'—':>10}{'—':>10}{'—':>10}  {r['error'][:50]}"); continue
    sp = r.get('s_per_step'); rel = f"{base_s/sp:.2f}x" if (sp and base_s) else "—"
    note = "❌ NaN / 未收斂" if r.get('nan') else ""
    print(f"{label:<18}{r['peak_gib']:>11.2f}G{sp:>10.3f}{rel:>10}"
          f"{(r.get('loss_first') or float('nan')):>10.4f}{(r.get('loss_last') or float('nan')):>10.4f}  {note}")

print()
print("要回答主管的三句話：")
print("  1. Week 2 用的是 MLX，不是純 transformers，也不是優化框架。")
print("  2. 這張表就是「有無記憶體優化」的量化差距。")
print("  3. loss 對得起來，代表省的記憶體不是靠犧牲數學換來的。")


---
# §5 LoRA 參數掃描

**這一節回答主管的 Q2。**

## 為什麼這是 Week 3 最重要的一節

翻 `mlx_lm/tuner/lora.py` 之後發現：

```python
return y + (self.scale * z).astype(x.dtype)     # scale 直接乘，沒有除以 rank
```

Week 2 用的 `scale: 20.0` 是**直接乘數**，換算成 PEFT 就是 `lora_alpha = 320`（r=16 → scaling 20）。業界常規是 `alpha=32, r=16` → scaling **2**。

**我們用的強度是常規的 10 倍，而且從來沒有意識到。**

Stage A 就是要驗證：Week 2 那 17.4 個百分點，有多少只是這一件事。

In [ ]:
#@title 5.1 通用訓練函式（含 logits 預估與 OOM 自動降規格）
import json, time, torch, gc
from pathlib import Path

VOCAB = 262144   # Gemma 3 的 vocab，logits 記憶體的元凶

def logits_gib(bs, seq, vocab=VOCAB):
    # fp16 的 logits + cross_entropy 內部的 fp32 副本 + 反向的 fp32 梯度
    e = bs * seq * vocab
    return {"fp16": e*2/2**30, "with_ce_fp32": e*6/2**30, "with_backward": e*10/2**30}

def _train_once(tag, *, model_name, data_file, rank, alpha, steps,
                target, seq, bs, ga, lr, seed, layers, save_dir):
    # 【坑：OOM 之後一定要在這個 scope 裡把 model / tr 刪掉】
    #   在呼叫端 except 裡收拾是沒用的 —— 例外的 traceback 會抓著這個 frame，
    #   frame 抓著 model，model 佔著 VRAM。下一次重試就會發現「記憶體還是滿的」，
    #   然後 bitsandbytes 會安靜地把塞不下的層丟到 CPU，最後拋一個看起來
    #   完全無關的 ValueError（Some modules are dispatched on the CPU or the disk）。
    #   所以整段包 try/finally，不管成功失敗都在這裡釋放。
    from unsloth import FastModel
    from trl import SFTTrainer
    from datasets import load_dataset
    try:

        ATTN = ["q_proj","k_proj","v_proj","o_proj"]
        MLP  = ["gate_proj","up_proj","down_proj"]
        tmods = ATTN if target == "attn" else ATTN + MLP

        reset_mem()
        base0 = baseline_gib()   # 同 session 殘留的 VRAM；peak_gib 會包含它
        require_vram(4.5, f"載入 {model_name} 的 4-bit 權重")
        t0 = time.time()
        model = tr = None
        model, tk = FastModel.from_pretrained(
            model_name=model_name, max_seq_length=seq, load_in_4bit=True, full_finetuning=False)
        model = FastModel.get_peft_model(
            model, r=rank, lora_alpha=alpha, lora_dropout=0.0, bias="none",
            target_modules=tmods, use_gradient_checkpointing="unsloth",
            random_state=seed, finetune_vision_layers=False, finetune_language_layers=True,
            **({"layers_to_transform": layers} if layers else {}))

        n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"[{tag}] 可訓練參數 {n_train/1e6:.2f}M | r={rank} alpha={alpha} "
              f"scaling={alpha/rank:.1f} | target={target} | seq={seq} bs={bs} ga={ga} | {steps} 步")

        ds = load_dataset("json", data_files=str(ROOT/'data'/data_file), split="train")
        timer = StepTimer()
        tr = SFTTrainer(model=model, train_dataset=ds, processing_class=tk,
            args=make_sft_config(output_dir=str(save_dir/'_ckpt'),
                           per_device_train_batch_size=bs, gradient_accumulation_steps=ga,
                           max_steps=steps, learning_rate=lr, warmup_ratio=0.05,
                           lr_scheduler_type="cosine", logging_steps=10, max_length=seq,
                           optim="adamw_8bit", fp16=False, bf16=False, report_to="none",
                           seed=seed, save_strategy="no", dataset_text_field="text"),
            callbacks=[timer.make()])
        tr.train()

        save_dir.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(str(save_dir)); tk.save_pretrained(str(save_dir))

        rec = timer.summary()
        rec.update({"tag": tag, "model": model_name, "data": data_file, "rank": rank,
                    "alpha": alpha, "scaling": alpha/rank, "target": target, "steps": steps,
                    "seq": seq, "bs": bs, "grad_accum": ga, "effective_batch": bs*ga, "lr": lr,
                    "trainable_params": n_train, "peak_gib": peak_gib(),
                    "baseline_gib": base0, "peak_delta_gib": peak_gib() - base0,
                    "logits_gib_est": logits_gib(bs, seq)["with_ce_fp32"],
                    "wall_min": (time.time()-t0)/60, "adapter_dir": str(save_dir)})
        return rec
    finally:
        # 真正的釋放：在持有變數的 scope 裡 del，再 gc + empty_cache。
        try: del tr
        except Exception: pass
        try: del model
        except Exception: pass
        gc.collect()
        try: torch.cuda.empty_cache()
        except Exception: pass
        print_vram(f"[{tag}] 釋放後")

def train_lora(tag, *, model_name, data_file, rank, alpha, steps,
               target="all", seq=1024, bs=1, ga=4, lr=1e-4, seed=42,
               layers=None, save_dir=None, auto_fit=True):
    # 【坑：T4 上真正的瓶頸是 logits，不是權重】
    #   gemma-3-4b 的 vocab 是 262,144。bs=2 × seq=1024 的 logits：
    #     fp16 1.0 GiB → cross_entropy 內部的 fp32 副本再 2.0 GiB → 反向再 2.0 GiB
    #   光這一項就 5 GiB，加上 4-bit 權重約 3 GiB 與活化，14.56 GiB 直接爆。
    #   這就是 Week 2 在 MLX 上撞到的同一個問題（logits 3.00 GiB 稀釋掉檢查點效益），
    #   只是換到 CUDA 上變成硬性的 OOM。
    #   → 預設改成 bs=1 / ga=4（等效 batch 仍是 4），並在 OOM 時自動降規格。
    if is_done(f"train_{tag}"):
        rec = json.loads(done_path(f"train_{tag}").read_text())
        adp = Path(rec.get("adapter_dir", ""))
        if (adp / "adapter_config.json").exists():
            print(f"[skip] {tag}（adapter 在 {adp}）")
            return rec
        # 紀錄還在但 adapter 不見了 —— 幾乎一定是執行階段被回收、
        # 而 adapter 存在 /content 上。重訓，不要讓後面的評測指到空路徑。
        print(f"[重訓] {tag}：完成紀錄存在，但 adapter 不見了（{adp}）")
        done_path(f"train_{tag}").unlink(missing_ok=True)

    save_dir = Path(save_dir or (ROOT/'out'/tag))    # ← Drive，不是 /content
    lg = logits_gib(bs, seq)
    print(f"[{tag}] logits 預估：fp16 {lg['fp16']:.2f} GiB → "
          f"含 CE 的 fp32 副本 {lg['with_ce_fp32']:.2f} GiB → 含反向 {lg['with_backward']:.2f} GiB")
    if lg["with_ce_fp32"] > 2.5:
        print(f"       ⚠️ 超過 2.5 GiB，T4 很可能撐不住。auto_fit 會在 OOM 時自動降。")

    plan = [(bs, seq)]
    if auto_fit:
        # 先砍 batch（不影響單筆樣本的完整性），再砍 seq（會截斷長樣本，是最後手段）
        if bs > 1: plan.append((1, seq))
        if seq > 512: plan.append((1, 512))
    tried = []
    for i, (b, sq) in enumerate(plan):
        try:
            rec = _train_once(tag, model_name=model_name, data_file=data_file, rank=rank,
                              alpha=alpha, steps=steps, target=target, seq=sq, bs=b,
                              ga=ga * (bs // b if b else 1), lr=lr, seed=seed,
                              layers=layers, save_dir=save_dir)
            rec["oom_retries"] = tried
            mark_done(f"train_{tag}", rec)
            return rec
        except torch.cuda.OutOfMemoryError as e:
            tried.append({"bs": b, "seq": sq, "error": str(e).split(".")[0]})
            print(f"\n  ❌ OOM at bs={b} seq={sq}")
            free_all(); torch.cuda.empty_cache(); gc.collect()
            if i == len(plan) - 1:
                raise RuntimeError(
                    f"{tag}: 降到 bs={b} seq={sq} 還是 OOM。\n"
                    f"  接下來可以試：target='attn'（少掉 gate/up/down）、"
                    f"seq=384、或升級 Colab Pro。") from e
            nb, nsq = plan[i+1]
            print(f"  → 自動降規格重試：bs={nb} seq={nsq}"
                  f"（logits {logits_gib(nb, nsq)['with_ce_fp32']:.2f} GiB）\n")

print("ok — 預設 bs=1 / ga=4（等效 batch 4），OOM 會自動降規格")


## Stage A — scaling 掃描（最重要的一組）

固定 `r=16`、all-linear、`inline` 資料、200 步，只動 `lora_alpha`。

| run | alpha | scaling | 意義 |
|---|---:|---:|---|
| A1 | 8 | 0.5 | 很保守 |
| A2 | 16 | 1.0 | |
| A3 | 32 | **2.0** | **業界常規** |
| A4 | 64 | 4.0 | |
| A5 | **320** | **20.0** | **重現 Week 2** |

**先寫下預測再跑**：無法解析率隨 scaling 單調上升，A5 應該重現 Week 2 的格式崩潰（~40%），A3 應該 < 5%。

In [ ]:
#@title 5.2 Stage A：scaling 掃描（約 5 × 12 分鐘）
STEPS_SWEEP = 200  #@param {type:"integer"}

STAGE_A = [("A1", 16, 8), ("A2", 16, 16), ("A3", 16, 32), ("A4", 16, 64), ("A5", 16, 320)]
# 手冊 §4.3：跑完框架對照必須重啟再跑 Step 5。這裡直接用程式擋，
# 忘了重啟就會在這裡停下來（fresh_required=True）。
require_stage_ready("Stage A", [n for n, _, _ in STAGE_A if not is_done(f"train_{n}")],
                    fresh_required=True)
for name, r_, a_ in STAGE_A:
    train_lora(name, model_name=MODEL_IT, data_file='train_inline.jsonl',
               rank=r_, alpha=a_, steps=STEPS_SWEEP, target="all")
print("\nStage A 完成。跑 §7 的評測才會有結論。")

In [ ]:
#@title 5.3 Stage B：rank 掃描（scaling 固定 2.0）
STAGE_B = [("B1", 8, 16), ("B2", 16, 32), ("B3", 32, 64), ("B4", 64, 128)]
require_stage_ready("Stage B", [n for n, _, _ in STAGE_B
                                if n != "B2" and not is_done(f"train_{n}")])
for name, r_, a_ in STAGE_B:
    if name == "B2":
        print("[skip] B2 == A3，同一組設定"); continue
    train_lora(name, model_name=MODEL_IT, data_file='train_inline.jsonl',
               rank=r_, alpha=a_, steps=STEPS_SWEEP, target="all")

In [ ]:
#@title 5.4 Stage C：target module（attn-only vs all-linear）
require_stage_ready("Stage C", [] if is_done("train_C1") else ["C1"])
train_lora("C1", model_name=MODEL_IT, data_file='train_inline.jsonl',
           rank=16, alpha=32, steps=STEPS_SWEEP, target="attn")
print("C2 == A3（all-linear），不重跑")


---
# §6 Shadow-FT

**這一節回答主管的 Q4。**

論文：*Shadow-FT: Tuning Instruct Model via Training on Paired Base Model*（arXiv 2505.12716）

```
Step 1:  W_B⁺ = Tune(W_B)                  ← 在 BASE 上訓練
Step 2:  W_I⁺ = W_I + (W_B⁺ − W_B)         ← 把差值搬到 INSTRUCT
```

**LoRA 下 base 項會抵消**：`W_I⁺ = W_I + (W_B + BA − W_B) = W_I + BA`。
所以實作就是「在 `-pt` 上訓練 adapter，然後把同一個 adapter 掛到 `-it` 上」。

## 2×2 設計

| | scaling = 2（常規） | scaling = 20（Week 2） |
|---|---|---|
| **常規 LoRA on `-it`** | A3 | A5 |
| **Shadow-FT（train on `-pt`）** | **D1** | **D2** |

- A5 崩、A3 不崩 → Week 2 的問題主要是超參數
- A3 也崩、D1 不崩 → 主管的判斷成立，Shadow-FT 是解法
- D1 和 D2 差距 << A3 和 A5 的差距 → **假設 S3：Shadow-FT 讓超參數不再那麼要命**（論文沒做這個）

**注意**：訓練 `-pt` 時，資料仍然是用 **`-it` 的 chat template** 渲染的（§2.1 已經做好了）。這一點很重要——delta 必須活在和目標模型同一個座標系裡。

In [ ]:
#@title 6.1 在 BASE 上訓練（D1 / D2）
require_stage_ready("Step 6 Shadow-FT",
                    [t for t in ("D1", "D2") if not is_done(f"train_{t}")])
train_lora("D1", model_name=MODEL_PT, data_file='train_inline.jsonl',
           rank=16, alpha=32,  steps=STEPS_SWEEP, target="all")
train_lora("D2", model_name=MODEL_PT, data_file='train_inline.jsonl',
           rank=16, alpha=320, steps=STEPS_SWEEP, target="all")


## 6.2 移植：把在 BASE 上學到的 delta 疊到 INSTRUCT

在 LoRA 底下，Shadow-FT 的 Eq.3 會退化成一件非常簡單的事：

```
W_I⁺ = W_I + (W_B + BA − W_B) = W_I + BA
```

**base 項互相抵消，所以「把在 `-pt` 上訓練的 adapter 直接掛到 `-it` 上」就是完整的 Shadow-FT。**
不需要把兩個模型的完整權重載進來做逐張量相減。

### 為什麼不融合成一份完整權重

融合會把 `W_I + BA` 實體化成一個新的 8.6 GB 模型。在免費版 Colab 上這條路走不通：

| | 系統 RAM | 磁碟 |
|---|---:|---:|
| 免費版總量 | **12.7 GB** | ~107 GB |
| 載入一個 fp16 shard 來改寫 | 8.6 GB | |
| 改寫時的 `.float()` 中間張量 | +2.7 GB | |
| 四個融合模型（A3/A5/D1/D2） | | 34 GB |

**8.6 + 2.7 已經逼近 12.7，kernel 會直接重啟。**

改成推論時掛 adapter：RAM 只需要 4-bit 的 base（約 3 GB），磁碟只需要 adapter（約 100 MB），
而且**常規 LoRA 和 Shadow-FT 走完全相同的程式路徑** —— 差別只在 adapter 是在 `-pt` 還是
`-it` 上訓練出來的。這反而比「兩邊都融合」更乾淨，因為連推論路徑的差異都消掉了。

In [ ]:
#@title 6.2 掛載 adapter = Shadow-FT 的移植（含「真的掛上去了」的對帳）
import json, torch
from pathlib import Path
from safetensors.torch import load_file

def load_for_eval(base_model, adapter_dir=None, load_4bit=True, max_seq=2048):
    # base_model  : 要當骨幹的模型（Shadow-FT 與常規 LoRA 都用 MODEL_IT）
    # adapter_dir : 要疊上去的 LoRA。None = 未微調的 baseline。
    #
    #   Shadow-FT = load_for_eval(MODEL_IT, D1 的 adapter)  ← adapter 在 -pt 上訓練
    #   常規 LoRA = load_for_eval(MODEL_IT, A3 的 adapter)  ← adapter 在 -it 上訓練
    # 兩者程式路徑完全相同。
    from unsloth import FastModel
    model, tk = FastModel.from_pretrained(
        model_name=base_model, max_seq_length=max_seq,
        load_in_4bit=load_4bit, full_finetuning=False)

    if adapter_dir:
        from peft import PeftModel
        adapter_dir = Path(adapter_dir)
        cfg = json.loads((adapter_dir / 'adapter_config.json').read_text())
        trained_on = cfg.get('base_model_name_or_path', '?')
        n_pairs = len({k.rsplit('.lora_', 1)[0]
                       for k in load_file(adapter_dir / 'adapter_model.safetensors')
                       if '.lora_' in k})

        model = PeftModel.from_pretrained(model, str(adapter_dir))

        # 【對帳】PEFT 對不上的模組是「靜默略過」的 —— 名稱不合就什麼都不會發生，
        #        不會報錯，只會得到一個和沒微調一模一樣的模型。這正是 Week 2
        #        k_proj/v_proj 一層都沒掛上卻毫無徵兆的同一種失敗。
        injected = len({n.rsplit('.lora_', 1)[0]
                        for n, _ in model.named_modules() if '.lora_A' in n})
        print(f"   adapter 訓練於 : {trained_on}")
        print(f"   骨幹模型       : {base_model}")
        print(f"   檔案裡的模組數 : {n_pairs}")
        print(f"   實際掛上的模組 : {injected}")
        assert injected > 0, (
            "adapter 一個模組都沒掛上去！target_modules 的名稱和骨幹模型對不起來。")
        assert injected == n_pairs, (
            f"只掛上 {injected}/{n_pairs} 個模組 —— 不要當作成功，先查名稱對應。")
        if trained_on != base_model:
            print(f"   → 這是 Shadow-FT：在 {trained_on} 上學的 delta 疊到 {base_model}")

    for_inference(model)
    return model, tk

print("ok — 之後所有評測都走這個函式")


## 6.3 四個要比較的組合

| 代號 | 骨幹 | adapter 訓練於 | scaling | 意義 |
|---|---|---|---:|---|
| `base_it` | `-it` | 無 | — | 未微調的基準線 |
| `A3` | `-it` | `-it` | 2.0 | 常規 LoRA，業界常規強度 |
| `A5` | `-it` | `-it` | 20.0 | 常規 LoRA，重現 Week 2 |
| `D1` | `-it` | **`-pt`** | 2.0 | **Shadow-FT** |
| `D2` | `-it` | **`-pt`** | 20.0 | **Shadow-FT**，Week 2 強度 |

`D1`/`D2` 的骨幹是 `-it`、adapter 是在 `-pt` 上訓練的 —— 這個組合本身就是 Shadow-FT，
不需要額外的合併步驟。

In [ ]:
#@title 6.3 列出四個組合，確認 adapter 都在
EVAL_COMBOS = [("base_it", MODEL_IT, None)]
for tag in ("A3", "A5", "D1", "D2"):
    if is_done(f"train_{tag}"):
        rec = json.loads(done_path(f"train_{tag}").read_text())
        EVAL_COMBOS.append((tag, MODEL_IT, rec['adapter_dir']))
        print(f"  {tag}: 骨幹 {MODEL_IT} + adapter {rec['adapter_dir']}"
              f"（訓練於 {rec['model']}, scaling {rec['scaling']:.1f}）")
    else:
        print(f"  {tag}: 尚未訓練，跳過")


## 6.4 （選用，只在 Colab Pro 以上跑）融合成獨立權重

只有在你需要一個**可以單獨發佈／部署**的模型時才需要這一步。做研究比較不需要 ——
6.2 的掛載方式在數學上完全等價（`W_I + BA`），而且更省。

**免費版不要跑這一格**：需要同時在 RAM 裡放一個 fp16 shard（8.6 GB）加上改寫用的中間張量，
12.7 GB 會不夠。下面的實作已經改成逐張量串流寫出，峰值 RAM 約等於**最大的單一張量**
（embed 約 1.3 GB），但 `save_file` 仍需把整個 shard 的字典握在手上。

In [ ]:
#@title 6.4 （選用）融合。免費版請保持 RUN_MERGE = False
RUN_MERGE = False  #@param {type:"boolean"}

import json, re, shutil, torch
from pathlib import Path
from safetensors.torch import load_file, save_file
from safetensors import safe_open
from huggingface_hub import snapshot_download

def shadow_graft(adapter_dir, target_repo, out_dir, alpha_scale=1.0):
    adapter_dir, out_dir = Path(adapter_dir), Path(out_dir)
    if (out_dir/'config.json').exists():
        print(f"[skip] {out_dir} 已存在"); return out_dir

    cfg = json.loads((adapter_dir/'adapter_config.json').read_text())
    lora_scaling = cfg['lora_alpha'] / cfg['r']
    if cfg.get('use_rslora'): lora_scaling = cfg['lora_alpha'] / (cfg['r'] ** 0.5)
    print(f"adapter r={cfg['r']} alpha={cfg['lora_alpha']} → scaling {lora_scaling}；移植 α={alpha_scale}")

    ad = load_file(adapter_dir/'adapter_model.safetensors')
    pairs = {}
    for k, v in ad.items():
        m = re.match(r'^(?:base_model\.model\.)?(.*)\.lora_(A|B)\.(?:default\.)?weight$', k)
        if m: pairs.setdefault(m.group(1), {})[m.group(2)] = v
    assert pairs, "adapter 裡沒有 lora_A/lora_B"
    print(f"解析出 {len(pairs)} 個模組")

    src = Path(snapshot_download(target_repo,
               allow_patterns=["*.safetensors", "*.json", "*.model", "*.jinja", "*.txt"]))
    out_dir.mkdir(parents=True, exist_ok=True)
    for f in src.iterdir():
        if f.is_file() and f.suffix in ('.json', '.model', '.txt', '.jinja'):
            shutil.copy(f, out_dir/f.name)

    grafted, stats = 0, []
    for f in sorted(src.glob('*.safetensors')):
        with safe_open(f, framework='pt') as h:
            keys = list(h.keys())
        sd = {}
        for k in keys:                       # 逐張量讀，不一次載入整個 shard
            with safe_open(f, framework='pt') as h:
                W = h.get_tensor(k)
            mod = k[:-len('.weight')] if k.endswith('.weight') else None
            hit = None
            if mod:
                for cand in (mod, mod.replace('model.', '', 1)):
                    if cand in pairs: hit = cand; break
                    m2 = [p for p in pairs if cand.endswith(p) or p.endswith(cand)]
                    if len(m2) == 1: hit = m2[0]; break
            if hit:
                A, B = pairs[hit]['A'].float(), pairs[hit]['B'].float()
                dW = (B @ A) * lora_scaling * alpha_scale
                assert dW.shape == W.shape, f"形狀不符 {hit}"
                rel = (dW.abs().sum() / W.float().abs().sum()).item()
                stats.append({"module": hit, "rel_delta": rel})
                W = (W.float() + dW).to(W.dtype)
                grafted += 1
                del A, B, dW
            sd[k] = W
        save_file(sd, str(out_dir/f.name), metadata={"format": "pt"})
        del sd; free_ram()
        print_ram(f"寫完 {f.name}")

    assert grafted == len(pairs), f"只移植了 {grafted}/{len(pairs)} 個模組，不要當作成功"
    rels = [x['rel_delta'] for x in stats]
    print(f"移植 {grafted} 個模組；‖ΔW‖/‖W‖ 平均 {sum(rels)/len(rels):.5f}"
          f" 最大 {max(rels):.5f}（應與 §3.2 的 σ 同數量級）")
    json.dump({"adapter": str(adapter_dir), "target": target_repo,
               "alpha_scale": alpha_scale, "n_grafted": grafted,
               "rel_delta_mean": sum(rels)/len(rels), "per_module": stats},
              open(ROOT/'reports'/f'graft_{out_dir.name}.json','w'), indent=2)
    return out_dir

if RUN_MERGE:
    for tag in ("D1", "D2", "A3", "A5"):
        if is_done(f"train_{tag}"):
            rec = json.loads(done_path(f"train_{tag}").read_text())
            shadow_graft(rec['adapter_dir'], MODEL_IT, SCRATCH/'out'/f'{tag}_merged')
else:
    print("RUN_MERGE = False。研究比較不需要融合，6.2 的掛載方式數學上等價。")


---
# §7 TMMLU+ 評測

## 為什麼不直接用 twinkle-eval

讀了 `Eval/twinkle_eval/` 的原始碼之後，發現兩件會影響數字可信度的事：

1. **`shuffle_options` 用的是沒設種子的全域 `random`**（整個套件 grep 不到任何 `seed`）。所以 Week 2 的 base 跑和 tuned 跑，**選項順序是不一樣的** —— 那 17.4 pt 裡有一部分是不同題目排列造成的雜訊。
2. **`average_accuracy` 是對「科目」取平均，不是對「題目」取平均。**三科題數 768 / 139 / 129，所以台語（129 題）和台灣地理（768 題）在總分裡權重相同。

下面這支評測器**完全複製 twinkle-eval 的 prompt 組法與 box 解析邏輯**（逐字比對過），但：

- 固定 shuffle 種子 → 每一組實驗看到**完全相同**的題目排列
- 同時報 **macro**（可和 Week 2 對照）和 **micro**（題目加權）
- 同時報 **嚴格**（只認 `\box{X}`）和 **寬鬆**（再接受「答案是 X」）
- 批次生成，比起 OpenAI 端點逐題呼叫快很多

In [ ]:
#@title 7.1 評測器（複製 twinkle-eval 的計分，但把種子固定）
import re, json, random, time, torch
import pandas as pd
from pathlib import Path

# ---- 與 twinkle-eval 的 BoxExtractor 逐字相同 ----
BOX_PATTERNS = [r"\\{1,2}box{([A-Z])}", r"\\{1,2}boxed{([A-Z])}"]
# ---- 與 scripts/analyze_eval.py 的寬鬆解析相同 ----
LENIENT = [r"box\{\s*([ABCD])\s*\}",
           r"(?:答案是|答案為|正確答案是|應該是|選項)\s*[:：]?\s*([ABCD])"]

def extract_strict(s):
    if not s: return None
    for p in BOX_PATTERNS:
        m = re.search(p, s)
        if m: return m.group(1).strip()
    return None

def extract_lenient(s):
    a = extract_strict(s)
    if a: return a
    if not s: return None
    for p in LENIENT:
        m = re.search(p, s)
        if m: return m.group(1).strip().upper()
    return None

def shuffle_options(row, rng):
    # 與 twinkle-eval 同樣「靠選項文字對回正解」，但 rng 由外部傳入 → 可重現。
    opts = [(k, row[k]) for k in "ABCD" if k in row and pd.notna(row[k])]
    if not opts: return None
    gold_text = row.get(str(row['answer']).strip().upper())
    rng.shuffle(opts)
    new = {"question": row['question']}
    for (old, text), newk in zip(opts, "ABCD"):
        new[newk] = text
        if text == gold_text: new['answer'] = newk
    return new if 'answer' in new else None

def build_prompt(q):
    # twinkle-eval evaluator.py:918 的組法
    return q['question'] + "\n" + "\n".join(f"{k}: {v}" for k, v in q.items()
                                            if k not in ("question", "answer"))

@torch.no_grad()
def evaluate(base_model, tag, adapter_dir=None, subjects=None, limit_per_subject=None,
             max_new_tokens=512, batch_size=8, seed=42, load_4bit=True):
    if is_done(f"eval_{tag}"):
        print(f"[skip] eval_{tag}")
        return json.loads(done_path(f"eval_{tag}").read_text())

    subjects = subjects or EVAL_SUBJECTS
    reset_mem(); t0 = time.time()
    # 走 §6.2 的 load_for_eval：骨幹固定是 MODEL_IT，差別只在掛哪個 adapter。
    model, tk = load_for_eval(base_model, adapter_dir, load_4bit=load_4bit)
    tk.padding_side = "left"
    if tk.pad_token is None: tk.pad_token = tk.eos_token

    per_subject, records = {}, []
    for s in subjects:
        df = pd.read_parquet(TMMLU_DIR / f"{s}.parquet")
        rng = random.Random(seed)          # ← 每一科都從同一個種子開始
        raw = [shuffle_options(r, rng) for _, r in df.iterrows()]
        qs = [q for q in raw if q]
        n_dropped = len(raw) - len(qs)
        if n_dropped:
            # twinkle-eval 會留著這些題；我們丟掉，所以要記下來，否則 n_questions 對不上 1,036
            print(f"    ⚠️ {s}: {n_dropped} 題因選項文字重複/缺失而無法對回正解，已剔除")
        if limit_per_subject: qs = qs[:limit_per_subject]

        n_ok_s = n_ok_l = n_unparsed = 0
        for i in range(0, len(qs), batch_size):
            batch = qs[i:i+batch_size]
            texts = []
            for q in batch:
                user = build_prompt(q)
                msgs = ([{"role":"system","content":SYS_BOX}] if HAS_SYSTEM else []) + \
                       [{"role":"user","content": (user if HAS_SYSTEM else SYS_BOX+"\n\n"+user)}]
                texts.append(tk.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))
            enc = tk(texts, return_tensors="pt", padding=True, truncation=True,
                     max_length=1536, add_special_tokens=False).to("cuda")
            out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                                 temperature=None, top_p=None, top_k=None,
                                 pad_token_id=tk.pad_token_id)
            gen = out[:, enc['input_ids'].shape[1]:]
            for q, g in zip(batch, gen):
                txt = tk.decode(g, skip_special_tokens=True)
                ps, pl = extract_strict(txt), extract_lenient(txt)
                ok_s, ok_l = (ps == q['answer']), (pl == q['answer'])
                n_ok_s += ok_s; n_ok_l += ok_l; n_unparsed += (ps is None)
                records.append({"subject": s, "question": q['question'][:200],
                                "gold": q['answer'], "pred_strict": ps, "pred_lenient": pl,
                                "correct_strict": bool(ok_s), "correct_lenient": bool(ok_l),
                                "n_gen_tokens": int((g != tk.pad_token_id).sum()),
                                "output": txt[:1500]})
            if i % (batch_size*10) == 0:
                print(f"    {s} {i+len(batch)}/{len(qs)}  嚴格 {n_ok_s/(i+len(batch)):.3f}  "
                      f"無法解析 {n_unparsed/(i+len(batch)):.3f}", flush=True)

        n = len(qs)
        per_subject[s] = {"n": n, "n_dropped": n_dropped,
                          "acc_strict": n_ok_s/n, "acc_lenient": n_ok_l/n,
                          "unparsed_rate": n_unparsed/n}
        print(f"  {s:<30} n={n:<5} 嚴格 {n_ok_s/n:.4f}  寬鬆 {n_ok_l/n:.4f}  "
              f"無法解析 {n_unparsed/n:.4f}")

    tot = sum(v['n'] for v in per_subject.values())
    res = {
        "tag": tag, "base_model": str(base_model),
        "adapter": str(adapter_dir) if adapter_dir else None,
        "n_questions": tot, "seed": seed,
        "macro_acc_strict":   sum(v['acc_strict']    for v in per_subject.values())/len(per_subject),
        "macro_acc_lenient":  sum(v['acc_lenient']   for v in per_subject.values())/len(per_subject),
        "macro_unparsed":     sum(v['unparsed_rate'] for v in per_subject.values())/len(per_subject),
        "micro_acc_strict":   sum(v['acc_strict']*v['n']    for v in per_subject.values())/tot,
        "micro_acc_lenient":  sum(v['acc_lenient']*v['n']   for v in per_subject.values())/tot,
        "micro_unparsed":     sum(v['unparsed_rate']*v['n'] for v in per_subject.values())/tot,
        "per_subject": per_subject, "minutes": (time.time()-t0)/60,
    }
    with (ROOT/'results'/f'eval_{tag}.jsonl').open('w') as f:
        for r in records: f.write(json.dumps(r, ensure_ascii=False) + "\n")
    mark_done(f"eval_{tag}", res)
    free_all(model)
    return res
print("ok")

In [ ]:
#@title 7.2 防呆：正式評測前先送 3 題，確認抽得出答案
#  Week 2 有一次評測請求全部成功、解析率卻是 0%，浪費了一整晚。
from unsloth import FastModel
import pandas as pd, random

def probe(base_model, adapter_dir=None, n=3):
    model, tk = load_for_eval(base_model, adapter_dir)
    tk.padding_side = "left"
    if tk.pad_token is None: tk.pad_token = tk.eos_token
    df = pd.read_parquet(TMMLU_DIR / f"{EVAL_SUBJECTS[0]}.parquet")
    rng = random.Random(42)
    qs = [q for q in (shuffle_options(r, rng) for _, r in df.head(n).iterrows()) if q]
    ok = 0
    for q in qs:
        user = build_prompt(q)
        msgs = ([{"role":"system","content":SYS_BOX}] if HAS_SYSTEM else []) + \
               [{"role":"user","content": (user if HAS_SYSTEM else SYS_BOX+"\n\n"+user)}]
        t = tk.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        enc = tk(t, return_tensors="pt", add_special_tokens=False).to("cuda")
        out = model.generate(**enc, max_new_tokens=512, do_sample=False, pad_token_id=tk.pad_token_id)
        txt = tk.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)
        got = extract_strict(txt)
        print(f"  正解 {q['answer']} | 抽到 {got} | 輸出前 120 字: {txt[:120]!r}")
        ok += got is not None
    free_all(model)
    assert ok > 0, "3 題一題都抽不出 \\box{} —— 先修 prompt，不要開跑正式評測"
    print(f"防呆通過（{ok}/{len(qs)} 題抽得出答案）")

# 探路過就跳過。它的目的是「開跑前確認抽得出 \\box{}」，
# 同一個模型在同一份 config 下確認過一次就夠了。
if is_done("probe_base_it"):
    print("[skip] 探路已通過（results/probe_base_it.json）")
else:
    probe(MODEL_IT)
    mark_done("probe_base_it", {"model": MODEL_IT, "note": "3 題都抽得出 box"})

In [ ]:
#@title 7.3 快速評測（每科 100 題）—— 用來掃參數
QUICK_N = 100  #@param {type:"integer"}

QUICK = [("base_it", MODEL_IT, None)]
for tag in ("A1","A2","A3","A4","A5","B1","B3","B4","C1","D1","D2"):
    if is_done(f"train_{tag}"):
        QUICK.append((tag, MODEL_IT, json.loads(done_path(f"train_{tag}").read_text())['adapter_dir']))

for tag, base, adp in QUICK:
    print(f"\n===== quick eval: {tag} =====")
    evaluate(base, f"quick_{tag}", adapter_dir=adp, limit_per_subject=QUICK_N, batch_size=8)
    free_ram(); print_ram()

In [ ]:
#@title 7.4 完整評測（1,036 題）—— 只跑決選的五個
for tag, base, adp in EVAL_COMBOS:
    print(f"\n===== full eval: {tag} =====")
    evaluate(base, f"full_{tag}", adapter_dir=adp, batch_size=8)
    free_ram(); print_ram()


---
# §8 彙整

In [ ]:
#@title 8.1 Stage A：scaling vs 格式保留（Week 3 的主圖）
import json, matplotlib
matplotlib.rcParams['axes.unicode_minus'] = False
import matplotlib.pyplot as plt

rows = []
for name, r_, a_ in STAGE_A:
    p = done_path(f"quick_{name}")
    if not p.exists(): continue
    e = json.loads(p.read_text()); t = json.loads(done_path(f"train_{name}").read_text())
    rows.append({"run": name, "scaling": t['scaling'], "alpha": a_,
                 "unparsed": e['micro_unparsed'], "strict": e['micro_acc_strict'],
                 "lenient": e['micro_acc_lenient'], "loss": t['loss_last']})

b = json.loads(done_path("quick_base_it").read_text()) if is_done("quick_base_it") else None

print(f"{'run':<5}{'alpha':>7}{'scaling':>9}{'無法解析':>10}{'嚴格':>9}{'寬鬆':>9}{'train loss':>12}")
if b: print(f"{'base':<5}{'—':>7}{'—':>9}{b['micro_unparsed']:>10.3f}"
            f"{b['micro_acc_strict']:>9.3f}{b['micro_acc_lenient']:>9.3f}{'—':>12}")
for r in rows:
    print(f"{r['run']:<5}{r['alpha']:>7}{r['scaling']:>9.1f}{r['unparsed']:>10.3f}"
          f"{r['strict']:>9.3f}{r['lenient']:>9.3f}{(r['loss'] or 0):>12.4f}")

if rows:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    x = [r['scaling'] for r in rows]
    ax[0].plot(x, [r['unparsed'] for r in rows], 'o-', color='crimson')
    if b: ax[0].axhline(b['micro_unparsed'], ls='--', c='gray', label='base (no FT)')
    ax[0].set_xscale('log'); ax[0].set_xlabel('LoRA scaling (alpha / r)')
    ax[0].set_ylabel('unparsed rate'); ax[0].set_title('format collapse vs LoRA strength')
    ax[0].legend(); ax[0].grid(alpha=.3)
    ax[1].plot(x, [r['strict'] for r in rows], 'o-', label='strict')
    ax[1].plot(x, [r['lenient'] for r in rows], 's-', label='lenient')
    if b:
        ax[1].axhline(b['micro_acc_strict'], ls='--', c='gray', label='base strict')
    ax[1].set_xscale('log'); ax[1].set_xlabel('LoRA scaling (alpha / r)')
    ax[1].set_ylabel('accuracy'); ax[1].set_title('accuracy vs LoRA strength')
    ax[1].legend(); ax[1].grid(alpha=.3)
    plt.tight_layout(); plt.savefig(ROOT/'reports'/'stageA_scaling.png', dpi=140)
    plt.show()
    print("\n判讀：")
    print("  無法解析率隨 scaling 單調上升 → Week 2 的崩潰主因是超參數。")
    print("  若 scaling=2 也崩            → 才是模型 post-training 的問題（Q3）。")

In [ ]:
#@title 8.2 2×2：常規 LoRA vs Shadow-FT
import json
grid = [("A3","常規 LoRA","2.0"), ("A5","常規 LoRA","20.0"),
        ("D1","Shadow-FT","2.0"), ("D2","Shadow-FT","20.0")]
print(f"{'方法':<12}{'scaling':>8}{'嚴格(micro)':>13}{'寬鬆':>9}{'無法解析':>10}{'嚴格(macro)':>13}")
if is_done("full_base_it"):
    b = json.loads(done_path("full_base_it").read_text())
    print(f"{'未微調 base':<12}{'—':>8}{b['micro_acc_strict']:>13.4f}"
          f"{b['micro_acc_lenient']:>9.4f}{b['micro_unparsed']:>10.4f}{b['macro_acc_strict']:>13.4f}")
for tag, meth, sc in grid:
    if not is_done(f"full_{tag}"): print(f"{meth:<12}{sc:>8}   （未跑）"); continue
    e = json.loads(done_path(f"full_{tag}").read_text())
    print(f"{meth:<12}{sc:>8}{e['micro_acc_strict']:>13.4f}{e['micro_acc_lenient']:>9.4f}"
          f"{e['micro_unparsed']:>10.4f}{e['macro_acc_strict']:>13.4f}")

print()
print("三條假設的判讀：")
print("  S1  Shadow-FT 的無法解析率 < 5%，且明顯低於同 scaling 的常規 LoRA")
print("  S2  Shadow-FT 的嚴格正確率 >= 未微調 base")
print("  S3  |D1 - D2| << |A3 - A5|  -> Shadow-FT 對超參數不敏感（論文沒做這一條）")

In [ ]:
#@title 8.3 匯出全部結果為一份 Markdown
import json, datetime
from pathlib import Path

lines = ["# Week 3 實驗結果（自動產生）", "",
         f"產生時間：{datetime.datetime.now():%Y-%m-%d %H:%M}", "",
         "> 所有數字由 `notebooks/week3_colab.ipynb` 產生，原始 JSON 在 `results/`。", ""]

def fmt(v):
    if v is None: return "—"
    if isinstance(v, float): return f"{v:.4f}"
    return str(v)

def table(title, tags, cols, hdr, notes=()):
    # 【坑】tags 必須和 mark_done 用的檔名一致：評測是 eval_quick_* / eval_full_*，
    #   不是 quick_* / full_*。第一輪就是這裡對不上，兩張評測表整個空掉。
    lines.append(f"## {title}"); lines.append("")
    lines.append("| " + " | ".join(hdr) + " |")
    lines.append("|" + "|".join(["---"] * len(hdr)) + "|")
    missing = []
    for t in tags:
        p = done_path(t)
        if not p.exists():
            missing.append(t); continue
        d = json.loads(p.read_text())
        lines.append("| " + " | ".join(fmt(d.get(c)) for c in cols) + " |")
    lines.append("")
    if missing:
        lines.append(f"（缺 {len(missing)} 筆：{', '.join(missing)}）"); lines.append("")
    for n in notes:
        lines.append(n)
    if notes: lines.append("")

table("框架對照（§4）", ["ab_hf_fp16", "ab_hf_fp32", "ab_unsloth"],
      ["framework", "peak_gib", "peak_delta_gib", "s_per_step", "loss_last", "nan", "error"],
      ["框架", "峰值 GiB", "淨峰值 GiB", "s/step", "末 loss", "NaN", "錯誤"])

table("訓練設定（§5–6）",
      [f"train_{t}" for t in ("A1","A2","A3","A4","A5","B1","B3","B4","C1","D1","D2")],
      ["tag","model","rank","alpha","scaling","target","trainable_params",
       "peak_gib","baseline_gib","peak_delta_gib","wall_min","loss_last"],
      ["run","模型","r","alpha","scaling","target","可訓練參數",
       "峰值 GiB","殘留 GiB","淨峰值 GiB","分鐘","末 loss"],
      notes=["> ⚠️ `峰值 GiB` 含同 session 之前累積的殘留 VRAM（每跑一組多 ~1.26 GiB），",
             "> 各 stage 越後面的組越虛高。引用記憶體數字請用 `淨峰值 GiB`（peak − 殘留）；",
             "> 第一輪的舊紀錄沒有這兩欄（顯示 —），只能引用各 session 第一組",
             "> （ab_unsloth 7.50 / B1 5.51 / D1 5.44）。"])

table("快速評測（每科 100 題）",
      [f"eval_quick_{t}" for t in ("base_it","A1","A2","A3","A4","A5","B1","B3","B4","C1","D1","D2")],
      ["tag","micro_acc_strict","micro_acc_lenient","micro_unparsed","macro_acc_strict"],
      ["run","嚴格(micro)","寬鬆(micro)","無法解析","嚴格(macro)"])

table("完整評測（1,036 題）",
      [f"eval_full_{t}" for t in ("base_it","A3","A5","D1","D2")],
      ["tag","n_questions","micro_acc_strict","micro_acc_lenient","micro_unparsed",
       "macro_acc_strict","minutes"],
      ["run","題數","嚴格(micro)","寬鬆(micro)","無法解析","嚴格(macro)","分鐘"])

_sig = ROOT / 'reports' / 'shadow_ft_sigma.json'
if _sig.exists():
    s = json.loads(_sig.read_text())
    lines += ["## Shadow-FT 前提 σ（§3.2）", "",
              f"- σ（全部張量，含 vision/projector）= {s['sigma']:.4f}",
              f"- σ（僅 language_model）= "
              + (f"{s['sigma_lm_only']:.4f}" if s.get('sigma_lm_only') is not None
                 else "未計算——重跑 §3.2 會補上"),
              f"- 論文門檻 < 0.05；判定用 language_model-only 那一個。", ""]

out = ROOT/'reports'/'week3_results.md'
out.write_text("\n".join(lines))
print(out); print(); print("\n".join(lines))



---
# §8.5 RAM 爆掉怎麼辦

免費版 Colab 的**系統 RAM 只有約 12.7 GB**，而且它和 GPU 的 15 GB VRAM 是分開的兩件事。
訓練吃的是 VRAM，但下面這些吃的是系統 RAM，而且更容易先爆：

| 動作 | 系統 RAM | 說明 |
|---|---:|---|
| 載入一份 gemma-3-4b 的 fp16 權重 | 8.6 GB | 只要一份就用掉三分之二 |
| 對它做 `.float()` | 再 +8.6 GB | fp16 → fp32 直接翻倍 |
| `(a - b)` 這種中間張量 | 再 +2.7 GB | embed 是 262,144 × 2,560 |

**這份 notebook 已經把兩個會爆的地方改掉了：**

- **§3.2 算 σ** —— 改用 `safe_open(...).get_slice()` 逐塊讀，峰值 RAM 只和 `CHUNK_ROWS` 有關，
  和張量大小無關。原本的 `get_tensor(k).float()` 是主要元凶。
- **§6 Shadow-FT** —— 不再融合成完整權重，改成推論時掛 adapter。
  數學上完全等價（`W_I⁺ = W_I + BA`），RAM 從 8.6 GB 降到 4-bit 骨幹的約 3 GB。

## kernel 已經重啟了，怎麼接回去

**不用從頭跑。**每個實驗跑完都會在 Drive 寫一個 `results/*.json`，重跑時已完成的會自動跳過。

1. 執行階段 → **重新啟動工作階段**（把殘留的記憶體清乾淨）
2. 從 §1.1 開始 Run All
3. 已完成的訓練與評測會印 `[skip]`，幾秒就過

## 還是爆的話，依序試

| 做法 | 怎麼做 |
|---|---|
| 跳過 σ 驗證 | §3.2 的 `RUN_SIGMA` 改 `False`。它只是驗證 Shadow-FT 的前提，不影響主線實驗 |
| 縮小分塊 | §3.2 的 `CHUNK_ROWS` 從 4096 降到 1024 |
| 清掉 fp16 快取 | §3.3 的 `DELETE_FP16` 改 `True`，省 17 GB 磁碟（σ 算完就用不到了） |
| 確認沒開融合 | §6.4 的 `RUN_MERGE` 必須是 `False` |

在任何一格前後插 `print_ram()` 就能看到是哪一步跳上去的。

---

# GPU VRAM（14.56 GiB）爆掉怎麼辦

這是另一件事。訊息長這樣：

```
OutOfMemoryError: CUDA out of memory. Tried to allocate 1.87 GiB.
  ... in torch._C._nn.cross_entropy_loss
```

## 為什麼是 cross_entropy 爆

**Gemma 3 的 vocab 是 262,144，logits 是這個模型上最大的單一張量。**

| bs × seq | fp16 logits | + CE 內部的 fp32 副本 | + 反向的梯度 |
|---|---:|---:|---:|
| 1 × 512 | 0.25 GiB | 0.75 GiB | 1.25 GiB |
| 1 × 1024 | 0.50 GiB | **1.50 GiB** | 2.50 GiB |
| 2 × 1024 | 1.00 GiB | **3.00 GiB** | **5.00 GiB** |
| 2 × 2048 | 2.00 GiB | 6.00 GiB | 10.00 GiB |

4-bit 權重約 3 GiB，加上活化與框架開銷，`bs=2 × seq=1024` 的 5 GiB logits 就會把
14.56 GiB 擠爆。**權重根本不是瓶頸，logits 才是。**

**這正是 Week 2 在 MLX 上撞到的同一個問題**（seq 2048 的 logits 吃掉 3.00 GiB，
把梯度檢查點的效益從 −91% 稀釋成 −30.5%），只是換到 CUDA 上從「效益被稀釋」
變成「直接 OOM」。梯度檢查點對 logits **完全無效** —— 它只作用在活化上。

## 已經做的三件事

1. **allocator 改成 `expandable_segments:True`**（§1.1，必須在 `import torch` 之前）。
   OOM 訊息裡的「1.19 GiB reserved but unallocated」就是碎片，這個設定專治它。
2. **`train_lora` 預設改成 `bs=1, ga=4`** —— 等效 batch 仍是 4，但 logits 減半。
3. **OOM 自動降規格**：先砍 batch，再砍 seq（512），每次重試都印出新的 logits 預估，
   而且會把降規格的過程記進結果 JSON 的 `oom_retries`，之後看表就知道哪幾組不是在
   同樣條件下跑的。

## 一個要寫進報告的觀察（Q1 相關）

traceback 停在 `torch._C._nn.cross_entropy_loss`，代表**完整的 logits 有落地** ——
Unsloth 的 fused / chunked cross-entropy 在這個組合下沒有生效。

這件事直接關係到主管問的 Q1：Unsloth 相對 HF 的主要賣點之一就是 fused CE
（不具現化完整 logits）。**如果它在 Gemma 3 上沒生效，那 §4 框架對照表裡
「峰值記憶體差多少」的解釋就要改寫** —— 省下來的不是 logits，是別的東西。

跑 §4 的時候留意三組的峰值記憶體差距：如果 Unsloth 和 HF fp32 的差距遠小於
logits 那 3 GiB，就證實 fused CE 沒有作用。**這是一個可以量的問題，不要用猜的。**

## 另一種長相：`ValueError: Some modules are dispatched on the CPU or the disk`

這個訊息**看起來和記憶體無關，其實就是 OOM 的下游症狀**。順序是：

1. 第一次嘗試 OOM
2. 前一次的模型**沒有被釋放**，還佔著 VRAM
3. 重試時 bitsandbytes 發現 GPU 塞不下，就「聰明地」把部分層丟到 CPU
4. 然後它自己的檢查擋下來，拋出這個和 OOM 完全不像的錯誤

**根因是我原本的清理方式沒有用。**`free_all(model, tr)` 收到的是新的區域參考，
`del o` 只刪掉函式內的迴圈變數，**呼叫端的 `model` 還在**。而 OOM 例外的
traceback 會抓著整個 frame，frame 抓著 model，model 抓著 VRAM。

現在 `_train_once` 整段包在 `try/finally` 裡，在**持有變數的那個 scope**
直接 `del model, tr`，不管成功或例外都會執行，並在每次重試前印出可用 VRAM。
另外 `require_vram()` 會在載入前先確認記憶體夠，不夠就直接給一個講得清楚的錯誤，
而不是讓 bitsandbytes 用一個無關的訊息把你導到錯的方向。

**如果還是遇到這個錯**：最可靠的解法就是「執行階段 → 重新啟動工作階段」再
從頭 Run All。已完成的會 `[skip]`，只有那一組會重跑。

## 還是 OOM 的話

| 做法 | 代價 |
|---|---|
| `target="attn"`（拿掉 gate/up/down） | 可訓練參數 29.8M → 8.9M，表達力下降，但 Stage C 本來就要跑這組 |
| `seq=512` | 資料的 p99 是 1,657 token，會截掉一部分長樣本 —— 要記進報告 |
| `steps` 減半 | 不影響記憶體，只影響訓練充分程度 |
| 升級 Colab Pro | L4 有 24 GB 且支援 bf16，上面所有問題一次消失 |


---
# §9 收尾檢查

跑完之後，**這幾個數字要能對得起來**，對不上就是哪裡錯了：

| 對帳項 | 條件 |
|---|---|
| `σ(base, instruct)`（§3.2） | < 0.05，否則 Shadow-FT 的前提不成立 |
| graft 的 `n_grafted` | **必須等於** adapter 裡的模組數，少一個都不行（程式已 assert） |
| `‖ΔW‖/‖W‖`（§6.2） | 應該和 σ 在同一個數量級。大很多代表 LoRA 太強了 |
| 未微調 base 的無法解析率 | < 5%。若一開始就高，是 prompt 有問題不是模型 |
| HF fp32 與 Unsloth 的 loss 曲線 | 應該接近。差很多代表某一邊數學不對 |
| 快速評測（100 題）與完整評測（1,036 題）的排序 | 應該一致。不一致代表 100 題的雜訊太大，掃描結論不可信 |

## 下載結果回本機

```bash
# 在本機 repo 執行
rsync -av ~/Google\ Drive/My\ Drive/ultrascale-lab-week3/results/  results/week3/
rsync -av ~/Google\ Drive/My\ Drive/ultrascale-lab-week3/reports/  reports/week3/
```

或直接用下一格打包下載。

In [ ]:
#@title 9.1 打包結果（不含大檔）
import shutil, os
pkg = '/content/week3_results'
shutil.rmtree(pkg, ignore_errors=True)
os.makedirs(pkg)
for d in ('results', 'reports'):
    shutil.copytree(ROOT/d, f'{pkg}/{d}', dirs_exist_ok=True)
shutil.make_archive('/content/week3_results', 'zip', pkg)
print("大小:", os.path.getsize('/content/week3_results.zip')/1e6, "MB")
try:
    from google.colab import files
    files.download('/content/week3_results.zip')
except Exception as e:
    print("自動下載失敗，從左側檔案面板手動下載 /content/week3_results.zip")